In [1]:
import torch
import math

# ============================================================
# Kernel Constants (from kernels.cu)
# ============================================================

MIN_VAL     = 1.17549435e-38   # FLT_MIN, kernel rsqrt floor
LOG_MIN_VAL = math.log2(MIN_VAL)  # ~ -126.0
INV_254     = 1.0 / 254.0
MIN_SCALE   = 1e-12

CLAMP_PRESETS = {
    "kernel (FLT_MIN)": LOG_MIN_VAL,
    "legacy (-53)":     -53.0,
    "adam_eps_equiv":   math.log2(1e-8),
}

# ============================================================
# Dynamic Map Codebook (Dettmers et al., 2022)
# Matches _create_dynamic_map in Adafactor8Bit Python source.
# ============================================================

def create_dynamic_map(signed=True, max_exponent_bits=7, total_bits=8):
    data = []
    non_sign_bits = total_bits - 1
    additional_items = 2 ** (non_sign_bits - max_exponent_bits) - 1
    for i in range(max_exponent_bits):
        fraction_items = int(
            2 ** (i + non_sign_bits - max_exponent_bits) + 1
            if signed
            else 2 ** (i + non_sign_bits - max_exponent_bits + 1) + 1
        )
        boundaries = torch.linspace(0.1, 1, fraction_items, dtype=torch.float32)
        means = (boundaries[:-1] + boundaries[1:]) / 2.0
        scale = 10 ** (-(max_exponent_bits - 1) + i)
        data += (scale * means).tolist()
        if signed:
            data += (-(scale * means)).tolist()
    if additional_items > 0:
        boundaries = torch.linspace(0.1, 1, additional_items + 1, dtype=torch.float32)
        means = (boundaries[:-1] + boundaries[1:]) / 2.0
        data += means.tolist()
        if signed:
            data += (-means).tolist()
    data.append(0)
    data.append(1.0)
    assert len(data) == 2 ** total_bits, f"Expected {2**total_bits}, got {len(data)}"
    data.sort()
    return torch.tensor(data, dtype=torch.float32)

BNB_U8 = create_dynamic_map(signed=False, total_bits=8)
BNB_S8 = create_dynamic_map(signed=True,  total_bits=8)
BNB_U4 = create_dynamic_map(signed=False, max_exponent_bits=3, total_bits=4)
BNB_S4 = create_dynamic_map(signed=True,  max_exponent_bits=3, total_bits=4)

NF4_LUT = torch.tensor([
    -1.0, -0.6961928009986877, -0.5250730514526367, -0.39491748809814453,
    -0.28444138169288635, -0.18477343022823334, -0.09105003625154495, 0.0,
    0.07958029955625534, 0.16093020141124725, 0.24611230194568634,
    0.33791524171829224, 0.44070982933044434, 0.5626170039176941,
    0.7229568362236023, 1.0,
], dtype=torch.float32)

# Sorted by value. Kernel stores in encoding order (0b000..0b111), which is not monotonic.
FP4_MAG_LUT = torch.tensor([
    0.0, 0.005208333333, 0.16666667, 0.25,
    0.33333333, 0.5, 0.66666667, 1.0,
], dtype=torch.float32)

# ============================================================
# Internal Helpers
# ============================================================

def _pad_blocks(v_flat, block_size):
    n = v_flat.numel()
    pad = (block_size - n % block_size) % block_size
    v_pad = torch.nn.functional.pad(v_flat, (0, pad)) if pad else v_flat
    return v_pad.view(-1, block_size), n

def _nearest_in_lut(values_flat, lut):
    """Nearest-neighbor lookup. For small LUTs only (k<=16); use searchsorted for k=256."""
    dists = (values_flat.unsqueeze(-1) - lut.unsqueeze(0)).abs()
    return lut[dists.argmin(dim=-1)]

# ============================================================
# V Quantization (quantize-dequantize roundtrip)
# ============================================================

def qd_log_adaptive(v, block_size, min_log_floor=LOG_MIN_VAL):
    """1+255 adaptive log-space, zero reserved. Kernel-aligned."""
    blocks, n = _pad_blocks(v.flatten(), block_size)
    is_zero = (blocks == 0)
    v_safe = blocks.clamp(min=MIN_VAL)
    log_v = torch.log2(v_safe)
    log_nz = log_v.clone()
    log_nz[is_zero] = float('inf')
    min_log = log_nz.min(dim=1, keepdim=True).values.clamp(min=min_log_floor)
    max_log = log_v.max(dim=1, keepdim=True).values.clamp(max=126.0)
    min_log = torch.where(min_log >= max_log, max_log - 1.0, min_log)
    scale = (max_log - min_log).clamp(min=MIN_SCALE)
    q = torch.round((log_v - min_log) / scale * 254.0 + 1.0).clamp(1, 255).to(torch.uint8)
    q[is_zero] = 0
    log_deq = (q.float() - 1.0) * scale / 254.0 + min_log
    result = torch.pow(2.0, log_deq)
    result[is_zero] = 0.0
    return result.flatten()[:n].view(v.shape)

def qd_log_adaptive_nz(v, block_size):
    """0+255 adaptive log-space, no zero reservation. Ablation."""
    blocks, n = _pad_blocks(v.flatten(), block_size)
    v_safe = blocks.clamp(min=MIN_VAL)
    log_v = torch.log2(v_safe)
    min_log = log_v.min(dim=1, keepdim=True).values
    max_log = log_v.max(dim=1, keepdim=True).values.clamp(max=126.0)
    scale = (max_log - min_log).clamp(min=MIN_SCALE)
    q = torch.round((log_v - min_log) / scale * 255.0).clamp(0, 255)
    return torch.pow(2.0, q * scale / 255.0 + min_log).flatten()[:n].view(v.shape)

def qd_log_fixed(v, block_size, min_log):
    """Fixed-floor log-space, 0+256 levels. Ablation."""
    min_log = max(min_log, LOG_MIN_VAL)
    blocks, n = _pad_blocks(v.flatten(), block_size)
    v_safe = blocks.clamp(min=2.0 ** min_log)
    log_v = torch.log2(v_safe)
    max_log = log_v.max(dim=1, keepdim=True).values.clamp(max=126.0)
    scale = (max_log - min_log).clamp(min=MIN_SCALE)
    q = torch.round((log_v - min_log) / scale * 255.0).clamp(0, 255)
    return torch.pow(2.0, q * scale / 255.0 + min_log).flatten()[:n].view(v.shape)

def qd_log_adaptive_generic(v, block_size, n_levels, zero_reserved=False,
                            min_log_floor=LOG_MIN_VAL):
    """Generic log-space: configurable levels and zero reservation."""
    blocks, n = _pad_blocks(v.flatten(), block_size)
    is_zero = (blocks == 0)
    v_safe = blocks.clamp(min=MIN_VAL)
    log_v = torch.log2(v_safe)
    if zero_reserved:
        log_nz = log_v.clone()
        log_nz[is_zero] = float('inf')
        min_log = log_nz.min(dim=1, keepdim=True).values.clamp(min=min_log_floor)
    else:
        min_log = log_v.min(dim=1, keepdim=True).values
    max_log = log_v.max(dim=1, keepdim=True).values.clamp(max=126.0)
    min_log = torch.where(min_log >= max_log, max_log - 1.0, min_log)
    scale = (max_log - min_log).clamp(min=MIN_SCALE)
    if zero_reserved:
        n_intervals = n_levels - 2
        q = torch.round((log_v - min_log) / scale * n_intervals + 1.0).clamp(1, n_levels - 1)
        q[is_zero] = 0
        log_deq = (q.float() - 1.0) / n_intervals * scale + min_log
        result = torch.pow(2.0, log_deq)
        result[is_zero] = 0.0
    else:
        q = torch.round((log_v - min_log) / scale * (n_levels - 1)).clamp(0, n_levels - 1)
        result = torch.pow(2.0, q * scale / (n_levels - 1) + min_log)
    return result.flatten()[:n].view(v.shape)

def qd_bnb(v, block_size):
    """Per-block absmax + unsigned 8-bit dynamic map (bnb 8-bit optimizer style for V)."""
    blocks, n = _pad_blocks(v.flatten(), block_size)
    qmap = BNB_U8.to(v.device)
    absmax = blocks.max(dim=1, keepdim=True).values.clamp(min=1e-30)
    normed = (blocks / absmax).flatten()
    idx = torch.searchsorted(qmap, normed.contiguous()).clamp(0, 255)
    prev = (idx - 1).clamp(0)
    idx = torch.where((normed - qmap[prev]).abs() < (normed - qmap[idx]).abs(), prev, idx)
    deq = qmap[idx] * absmax.squeeze(-1).repeat_interleave(block_size)
    return deq[:n].view(v.shape)

def qd_dynmap(v, block_size, bits=4):
    """Dynamic map (unsigned), generic bits. For V ablation (not a bnb product)."""
    qmap = (BNB_U4 if bits == 4 else BNB_U8).to(v.device)
    n_levels = len(qmap)
    blocks, n = _pad_blocks(v.flatten(), block_size)
    absmax = blocks.max(dim=1, keepdim=True).values.clamp(min=1e-30)
    normed = (blocks / absmax).flatten()
    idx = torch.searchsorted(qmap, normed.contiguous()).clamp(0, n_levels - 1)
    prev = (idx - 1).clamp(0)
    idx = torch.where((normed - qmap[prev]).abs() < (normed - qmap[idx]).abs(), prev, idx)
    deq = qmap[idx] * absmax.squeeze(-1).repeat_interleave(block_size)
    return deq[:n].view(v.shape)

def qd_uniform(v, block_size, n_levels=256):
    """Uniform linear quantization for non-negative V."""
    blocks, n = _pad_blocks(v.flatten(), block_size)
    absmax = blocks.max(dim=1, keepdim=True).values.clamp(min=1e-30)
    normed = blocks / absmax
    q = torch.round(normed * (n_levels - 1)).clamp(0, n_levels - 1)
    return (q / (n_levels - 1) * absmax).flatten()[:n].view(v.shape)

def qd_nf4_v(v, block_size):
    """NF4 LUT adapted for non-negative V (weight-designed, used as control)."""
    blocks, n = _pad_blocks(v.flatten(), block_size)
    absmax = blocks.max(dim=1, keepdim=True).values.clamp(min=1e-30)
    normed = (blocks / absmax).clamp(0, 1).flatten()
    lut_pos = NF4_LUT[NF4_LUT >= 0].to(v.device)
    deq = _nearest_in_lut(normed, lut_pos) * absmax.squeeze(-1).repeat_interleave(block_size)
    return deq[:n].view(v.shape)

def qd_fp4_v(v, block_size):
    """FP4 magnitude LUT adapted for non-negative V (weight-designed, used as control)."""
    blocks, n = _pad_blocks(v.flatten(), block_size)
    absmax = blocks.max(dim=1, keepdim=True).values.clamp(min=1e-30)
    normed = (blocks / absmax).clamp(0, 1).flatten()
    lut = FP4_MAG_LUT.to(v.device)
    deq = _nearest_in_lut(normed, lut) * absmax.squeeze(-1).repeat_interleave(block_size)
    return deq[:n].view(v.shape)

# ============================================================
# M Quantization (signed, quantize-dequantize roundtrip)
# ============================================================

def qd_uf(v, block_size, bits):
    """Uniform symmetric quantization for M. Kernel m_mode=0."""
    blocks, n = _pad_blocks(v.flatten(), block_size)
    absmax = blocks.abs().max(dim=1, keepdim=True).values.clamp(min=1e-12)
    half = 2 ** (bits - 1)
    q = torch.round(blocks / absmax * half).long() + half
    q = q.clamp(0, 2 ** bits - 1)
    deq = (q.float() - half) * (absmax / half)
    return deq.flatten()[:n].view(v.shape)

def qd_dynamic_signed(v, block_size, bits):
    """Dynamic map signed quantization for M. Kernel m_mode=1."""
    qmap = (BNB_S4 if bits == 4 else BNB_S8).to(v.device)
    n_levels = len(qmap)
    blocks, n = _pad_blocks(v.flatten(), block_size)
    absmax = blocks.abs().max(dim=1, keepdim=True).values.clamp(min=1e-12)
    x_norm = (blocks / absmax).flatten()
    x_flat = blocks.flatten()
    idx = torch.searchsorted(qmap, x_norm.contiguous()).clamp(0, n_levels - 1)
    prev = (idx - 1).clamp(0)
    idx = torch.where((x_norm - qmap[prev]).abs() < (x_norm - qmap[idx]).abs(), prev, idx)
    map_neg = torch.signbit(qmap[idx])
    val_neg = torch.signbit(x_flat)
    mismatch = (map_neg != val_neg) & (x_flat != 0.0)
    idx = torch.where(mismatch & (x_flat > 0), (idx + 1).clamp(max=n_levels - 1), idx)
    idx = torch.where(mismatch & (x_flat < 0), (idx - 1).clamp(min=0), idx)
    deq = qmap[idx] * absmax.squeeze(-1).repeat_interleave(block_size)
    return deq[:n].view(v.shape)

# ============================================================
# Metrics
# ============================================================

def rel_l2(v_true, v_approx):
    """Relative L2 error on non-zero elements."""
    mask = v_true > 0
    if mask.sum() == 0:
        return 0.0
    t = v_true[mask].double()
    a = v_approx[mask].double()
    return ((t - a).norm() / t.norm()).item()

def rel_l2_signed(v_true, v_approx):
    """Relative L2 error for signed tensors (M)."""
    ref = v_true.norm()
    if ref < 1e-30:
        return 0.0
    return ((v_true - v_approx).norm() / ref).item()

def update_error_adam(grad, v_true, v_approx, eps=1e-8, d=0.0):
    """Update error, Adam-style: 1/(sqrt(v) + eps). d>0 clips ref and approx independently."""
    u_ref = grad / (v_true.sqrt() + eps)
    u_app = grad / (v_approx.sqrt() + eps)
    if d > 0:
        N = grad.numel()
        u_ref = u_ref / max((u_ref.norm() / math.sqrt(N)).item() / d, 1.0)
        u_app = u_app / max((u_app.norm() / math.sqrt(N)).item() / d, 1.0)
    ref_n = u_ref.norm()
    return ((u_ref - u_app).norm() / ref_n).item() if ref_n > 1e-30 else 0.0

def update_error_adafactor(grad, v_true, v_approx, d=0.0):
    """Update error, Adafactor-style: rsqrt(fmaxf(v, MIN_VAL)). d>0 clips ref and approx independently."""
    u_ref = grad * torch.rsqrt(v_true.clamp(min=MIN_VAL))
    u_app = grad * torch.rsqrt(v_approx.clamp(min=MIN_VAL))
    if d > 0:
        N = grad.numel()
        u_ref = u_ref / max((u_ref.norm() / math.sqrt(N)).item() / d, 1.0)
        u_app = u_app / max((u_app.norm() / math.sqrt(N)).item() / d, 1.0)
    ref_n = u_ref.norm()
    return ((u_ref - u_app).norm() / ref_n).item() if ref_n > 1e-30 else 0.0

def update_error_no_clamp(grad, v_true, v_approx):
    """Update error, kernel NOCLIP path: v>0 ? rsqrt(v) : 0."""
    inv_ref = torch.zeros_like(v_true)
    inv_app = torch.zeros_like(v_approx)
    mask_ref = v_true > 0
    mask_app = v_approx > 0
    inv_ref[mask_ref] = torch.rsqrt(v_true[mask_ref])
    inv_app[mask_app] = torch.rsqrt(v_approx[mask_app])
    u_ref = grad * inv_ref
    u_app = grad * inv_app
    ref_n = u_ref.norm()
    return ((u_ref - u_app).norm() / ref_n).item() if ref_n > 1e-30 else 0.0

def clamp_distortion(grad, v_true):
    """Distortion from MIN_VAL floor vs no floor. Skips V=0 elements."""
    mask = v_true > 0
    if mask.sum() == 0:
        return 0.0
    g, v = grad[mask], v_true[mask]
    u_no = g * torch.rsqrt(v.clamp(min=1e-30))
    u_cl = g * torch.rsqrt(v.clamp(min=MIN_VAL))
    norm = u_no.norm()
    return ((u_no - u_cl).norm() / norm).item() if norm > 1e-30 else 0.0

def inv_std_error_percentiles(v_true, v_approx):
    """Per-element inv_std relative error percentiles (Adafactor path)."""
    inv_t = torch.rsqrt(v_true.clamp(min=MIN_VAL))
    inv_a = torch.rsqrt(v_approx.clamp(min=MIN_VAL))
    mask = inv_t > 1e-30
    if mask.sum() == 0:
        return 0.0, 0.0, 0.0
    rel = ((inv_a[mask] - inv_t[mask]).abs() / inv_t[mask])
    return rel.median().item(), rel.quantile(0.95).item(), rel.quantile(0.99).item()

def cosine_sim(a, b):
    na, nb = a.norm(), b.norm()
    if na < 1e-30 or nb < 1e-30:
        return 0.0
    return ((a * b).sum() / (na * nb)).item()

def sign_flip_rate(true, approx):
    mask = true.abs() > 1e-30
    if mask.sum() == 0:
        return 0.0
    flips = (true[mask].sign() != approx[mask].sign()) & (approx[mask].abs() > 1e-30)
    return flips.float().mean().item()

def asymmetry_bias(true, approx):
    mask = true.abs() > 1e-30
    if mask.sum() == 0:
        return 0.0
    return ((approx[mask] - true[mask]) / true[mask].abs()).mean().item()

# ============================================================
# Distribution Generators
# ============================================================

def simulate_v_ema(grad_std, beta, steps, numel=2048, sparsity=0.0, seed=None):
    rng = torch.Generator().manual_seed(seed if seed is not None else 0)
    v = torch.zeros(numel)
    cold_mask = torch.rand(numel, generator=rng) < sparsity
    for _ in range(steps):
        g = torch.randn(numel, generator=rng) * grad_std
        g[cold_mask] = 0.0
        v = (1.0 - beta) * v + beta * g.square()
    return v

def gen_v(name, n, seed=42):
    rng = torch.Generator().manual_seed(seed)
    if name == "narrow_0.5dec":
        return torch.pow(2.0, torch.randn(n, generator=rng) * 0.15 - 13.3)
    elif name == "narrow_3dec":
        return torch.pow(2.0, torch.randn(n, generator=rng) * 1.0 - 13.3)
    elif name == "medium_10dec":
        return torch.pow(2.0, torch.randn(n, generator=rng) * 3.0 - 15.0)
    elif name == "wide_20dec":
        return torch.pow(2.0, torch.randn(n, generator=rng) * 6.0 - 20.0)
    elif name == "wide_35dec":
        return torch.pow(2.0, torch.randn(n, generator=rng) * 10.0 - 25.0)
    elif name == "outlier":
        v = torch.pow(2.0, torch.randn(n, generator=rng) * 0.5 - 13.3)
        n_out = max(1, n // 100)
        v[:n_out] = torch.pow(2.0, torch.randn(n_out, generator=rng) * 1.0 - 3.0)
        return v
    elif name == "bimodal":
        n_hot = n // 5
        v = torch.zeros(n)
        v[:n_hot] = torch.pow(2.0, torch.randn(n_hot, generator=rng) * 0.5 - 10.0)
        v[n_hot:] = torch.pow(2.0, torch.randn(n - n_hot, generator=rng) * 0.5 - 30.0)
        return v
    elif name == "ema_realistic":
        v = torch.zeros(n)
        for _ in range(500):
            g = torch.randn(n, generator=rng) * 0.01
            v = 0.999 * v + 0.001 * g.square()
        return v
    elif name == "dense_warm":
        return simulate_v_ema(0.01, 0.001, 1000, n, seed=seed)
    elif name == "dense_hot":
        return simulate_v_ema(0.1, 0.001, 1000, n, seed=seed)
    elif name == "dense_cold_start":
        return simulate_v_ema(0.01, 0.001, 10, n, seed=seed)
    elif name == "sparse_40pct":
        v = torch.zeros(n)
        n_hot = int(n * 0.6)
        v[:n_hot] = torch.pow(2.0, torch.randn(n_hot, generator=rng) * 1.0 - 13.3)
        return v
    elif name == "sparse_90pct":
        v = torch.zeros(n)
        n_hot = n // 10
        v[:n_hot] = torch.pow(2.0, torch.randn(n_hot, generator=rng) * 0.5 - 13.3)
        return v
    elif name == "wide_range":
        v = torch.zeros(n)
        q = n // 4
        v[:q] = 1e-2; v[q:2*q] = 1e-5; v[2*q:3*q] = 1e-10
        return v
    elif name == "uniform_small":
        return torch.rand(n, generator=rng) * 1e-6 + 1e-12
    else:
        raise ValueError(f"Unknown distribution: {name}")

def gen_ada_v(name, R=64, C=128, steps=500, beta2=0.999, grad_std=0.01, seed=42):
    """Generate Adafactor-style factored V (row * col / row_mean)."""
    rng = torch.Generator().manual_seed(seed)
    if name == "ada_dense":
        active = torch.ones(R, dtype=torch.bool)
    elif name == "ada_mixed":
        active = torch.arange(R) < int(R * 0.4)
    elif name == "ada_cold":
        active = torch.arange(R) < int(R * 0.1)
    else:
        raise ValueError(f"Unknown: {name}")
    row_sum = torch.zeros(R)
    col_sum = torch.zeros(C)
    for _ in range(steps):
        g = torch.randn(R, C, generator=rng) * grad_std
        g[~active] = 0.0
        g_sq = g.square()
        row_sum = beta2 * row_sum + (1 - beta2) * g_sq.mean(dim=1)
        col_sum = beta2 * col_sum + (1 - beta2) * g_sq.mean(dim=0)
    row_mean_val = row_sum.mean().clamp(min=1e-30)
    V = row_sum.unsqueeze(1) * col_sum.unsqueeze(0) / row_mean_val
    return V.flatten()

def gen_m_ema(n, steps, beta1, grad_std, seed=42):
    rng = torch.Generator().manual_seed(seed)
    m = torch.zeros(n)
    for _ in range(steps):
        g = torch.randn(n, generator=rng) * grad_std
        m = beta1 * m + (1 - beta1) * g
    return m

def fmt_err(val):
    """Format percentage, capping display at >100%."""
    return f"{val:>9.3f}" if val <= 100 else "    >100%"

ALL_V_DISTS = [
    "narrow_0.5dec", "narrow_3dec", "medium_10dec", "wide_20dec",
    "wide_35dec", "outlier", "bimodal", "ema_realistic", "sparse_90pct",
    "dense_warm", "dense_hot", "dense_cold_start", "wide_range", "uniform_small",
]

ALL_ADA_DISTS = ["ada_dense", "ada_mixed", "ada_cold"]

print(f"§0 loaded. Codebooks: U8={len(BNB_U8)}, S8={len(BNB_S8)}, "
      f"U4={len(BNB_U4)}, S4={len(BNB_S4)}, "
      f"NF4={len(NF4_LUT)}, FP4={len(FP4_MAG_LUT)}")

§0 loaded. Codebooks: U8=256, S8=256, U4=16, S4=16, NF4=16, FP4=8


In [2]:
# ============================================================
# Codebook Verification
# ============================================================

# --- Structural properties (algorithm-independent) ---

def check_codebook_properties(name, cb, signed, expected_len):
    assert len(cb) == expected_len, f"{name}: length {len(cb)} != {expected_len}"
    assert torch.all(cb[1:] >= cb[:-1]), f"{name}: not sorted"
    assert (cb == 0).sum() == 1, f"{name}: expected exactly one zero"
    if signed:
        assert cb[0].item() < 0, f"{name}: smallest should be negative"
        assert cb[-1].item() == 1.0, f"{name}: largest should be 1.0"
        # map is symmetric except for the appended +1.0 (no -1.0 counterpart)
        assert abs(cb[0].item() + cb[-2].item()) < 1e-6, \
            f"{name}: not symmetric around zero (cb[0]={cb[0].item()}, cb[-2]={cb[-2].item()})"
    else:
        assert cb[0].item() == 0.0, f"{name}: smallest should be 0.0"
        assert cb[-1].item() == 1.0, f"{name}: largest should be 1.0"
    print(f"  {name}: PASS ({expected_len} entries, properties hold)")

check_codebook_properties("unsigned 8-bit (V)", BNB_U8, signed=False, expected_len=256)
check_codebook_properties("signed   8-bit (M)", BNB_S8, signed=True,  expected_len=256)
check_codebook_properties("unsigned 4-bit",     BNB_U4, signed=False, expected_len=16)
check_codebook_properties("signed   4-bit (M)", BNB_S4, signed=True,  expected_len=16)

# --- Cross-check against independent re-implementation ---

def _ref_dynamic_map(signed, max_exp, total):
    data = []
    non_sign_bits = total - 1
    additional_items = 2 ** (non_sign_bits - max_exp) - 1
    for i in range(max_exp):
        frac = int(
            2 ** (i + non_sign_bits - max_exp) + 1
            if signed
            else 2 ** (i + non_sign_bits - max_exp + 1) + 1
        )
        boundaries = torch.linspace(0.1, 1, frac, dtype=torch.float32)
        means = (boundaries[:-1] + boundaries[1:]) / 2.0
        s = 10 ** (-(max_exp - 1) + i)
        data += (s * means).tolist()
        if signed:
            data += (-(s * means)).tolist()
    if additional_items > 0:
        boundaries = torch.linspace(0.1, 1, additional_items + 1, dtype=torch.float32)
        means = (boundaries[:-1] + boundaries[1:]) / 2.0
        data += means.tolist()
        if signed:
            data += (-means).tolist()
    data.append(0)
    data.append(1.0)
    data.sort()
    return torch.tensor(data, dtype=torch.float32)

cross_checks = [
    ("U8 cross-check", BNB_U8, _ref_dynamic_map(False, 7, 8)),
    ("S8 cross-check", BNB_S8, _ref_dynamic_map(True,  7, 8)),
    ("U4 cross-check", BNB_U4, _ref_dynamic_map(False, 3, 4)),
    ("S4 cross-check", BNB_S4, _ref_dynamic_map(True,  3, 4)),
]
for name, actual, expected in cross_checks:
    assert torch.allclose(actual, expected, atol=0), f"{name}: mismatch"
    print(f"  {name}: PASS")

# --- Kernel LUT verification ---

nf4_ref = torch.tensor([
    -1.0, -0.6961928009986877, -0.5250730514526367, -0.39491748809814453,
    -0.28444138169288635, -0.18477343022823334, -0.09105003625154495, 0.0,
    0.07958029955625534, 0.16093020141124725, 0.24611230194568634,
    0.33791524171829224, 0.44070982933044434, 0.5626170039176941,
    0.7229568362236023, 1.0,
])
assert torch.allclose(NF4_LUT, nf4_ref, atol=0), "NF4 LUT mismatch"
nf4_nonneg = (NF4_LUT >= 0).sum().item()
print(f"  NF4 LUT: PASS ({len(NF4_LUT)} entries, {nf4_nonneg} non-negative)")

fp4_ref = torch.tensor([0.0, 0.005208333333, 0.16666667, 0.25,
                        0.33333333, 0.5, 0.66666667, 1.0])
assert torch.allclose(FP4_MAG_LUT, fp4_ref, atol=0), "FP4 LUT mismatch"
print(f"  FP4 MAG LUT: PASS ({len(FP4_MAG_LUT)} magnitude levels)")

print("\nAll codebook checks passed.")

  unsigned 8-bit (V): PASS (256 entries, properties hold)
  signed   8-bit (M): PASS (256 entries, properties hold)
  unsigned 4-bit: PASS (16 entries, properties hold)
  signed   4-bit (M): PASS (16 entries, properties hold)
  U8 cross-check: PASS
  S8 cross-check: PASS
  U4 cross-check: PASS
  S4 cross-check: PASS
  NF4 LUT: PASS (16 entries, 9 non-negative)
  FP4 MAG LUT: PASS (8 magnitude levels)

All codebook checks passed.


## §A Design Space Exploration

Why V quantization needs adaptive min_log + 1+255.
§E isolates each design choice; §A shows the full landscape.

In [3]:
# A1: Fixed min_log sweep + three-layer decomposition
# L1: pure quantization error (non-zero elements, no clamp)
# L2: update error with kernel clamp (rsqrt + MIN_VAL)
# L3: clamp distortion (independent of min_log, computed once per distribution)

min_logs = [-126, -80, -60, -53, -44, -40, -36, -32, -28, -24, -20]
dists = ["narrow_0.5dec", "medium_10dec", "wide_20dec", "sparse_90pct"]
bs = 2048

# --- Part 1: Three-layer decomposition ---
print("A1a: Three-layer decomposition (block_size=2048)")
print("  L1=quant error, L2=update error (kernel clamp), L3=clamp distortion")

methods_3l = {
    "adaptive 1+255": lambda v: qd_log_adaptive(v, bs),
    "adaptive 0+255": lambda v: qd_log_adaptive_nz(v, bs),
    "fixed -126":     lambda v: qd_log_fixed(v, bs, -126.0),
    "fixed -40":      lambda v: qd_log_fixed(v, bs, -40.0),
}

print(f"\n  {'distribution':<18} {'L3_clamp%':>10}")
print("  " + "-" * 30)
for dname in dists:
    v = gen_v(dname, 8192, seed=42)
    rng = torch.Generator().manual_seed(7)
    grad = torch.randn(8192, generator=rng) * 0.01
    l3 = clamp_distortion(grad, v) * 100
    print(f"  {dname:<18} {l3:>9.4f}")

print(f"\n  {'distribution':<18} {'method':<18} {'L1_qnt%':>9} {'L2_upd%':>9}")
print("  " + "-" * 58)
for dname in dists:
    v = gen_v(dname, 8192, seed=42)
    rng = torch.Generator().manual_seed(7)
    grad = torch.randn(8192, generator=rng) * 0.01
    for mname, mfn in methods_3l.items():
        v_q = mfn(v)
        l1 = rel_l2(v, v_q) * 100
        l2 = update_error_adafactor(grad, v, v_q) * 100
        print(f"  {dname:<18} {mname:<18} {l1:>8.3f} {fmt_err(l2)}")
    print()

# --- Part 2: Fine min_log sweep (L2 metric) ---
print("A1b: Fixed min_log sweep (L2 update error, kernel clamp)")
print(f"  {'distribution':<18}", end="")
for ml in min_logs:
    print(f" {ml:>6}", end="")
print(f" {'adapt':>8}")
print("  " + "-" * (18 + 7 * len(min_logs) + 9))

for dname in dists:
    v = gen_v(dname, 8192, seed=42)
    rng = torch.Generator().manual_seed(7)
    grad = torch.randn(8192, generator=rng) * 0.01
    print(f"  {dname:<18}", end="")
    for ml in min_logs:
        v_q = qd_log_fixed(v, bs, float(ml))
        l2 = update_error_adafactor(grad, v, v_q) * 100
        print(f" {l2:>5.2f}%", end="")
    v_q = qd_log_adaptive(v, bs)
    l2 = update_error_adafactor(grad, v, v_q) * 100
    print(f" {l2:>7.3f}%")

A1a: Three-layer decomposition (block_size=2048)
  L1=quant error, L2=update error (kernel clamp), L3=clamp distortion

  distribution        L3_clamp%
  ------------------------------
  narrow_0.5dec         0.0000
  medium_10dec          0.0000
  wide_20dec            0.0000
  sparse_90pct          0.0000

  distribution       method               L1_qnt%   L2_upd%
  ----------------------------------------------------------
  narrow_0.5dec      adaptive 1+255        0.079     0.039
  narrow_0.5dec      adaptive 0+255        0.078     0.039
  narrow_0.5dec      fixed -126            8.430     4.200
  narrow_0.5dec      fixed -40             2.104     1.052

  medium_10dec       adaptive 1+255        1.373     0.688
  medium_10dec       adaptive 0+255        1.367     0.701
  medium_10dec       fixed -126            8.227     5.140
  medium_10dec       fixed -40             2.336     1.574

  wide_20dec         adaptive 1+255        1.526     0.704
  wide_20dec         adaptive 0+255 

In [4]:
# A2: Stability of min_log sweep (10 seeds)
# Tests whether the sweep results are seed-dependent.

min_logs_test = [-126, -53, -40, -32, -24]
dists = ["narrow_0.5dec", "medium_10dec", "wide_20dec"]
n_trials = 10
bs = 2048

print(f"A2: Stability ({n_trials} seeds, L2 update error %)")

for dname in dists:
    print(f"\n  {dname}")
    print(f"  {'min_log':<8} {'mean%':>9} {'std%':>9} {'min%':>9} {'max%':>9}")
    print("  " + "-" * 48)
    for ml in min_logs_test:
        errs = []
        for trial in range(n_trials):
            v = gen_v(dname, 8192, seed=trial * 100 + 7)
            rng = torch.Generator().manual_seed(trial * 100 + 8)
            grad = torch.randn(8192, generator=rng) * 0.01
            v_q = qd_log_fixed(v, bs, float(ml))
            errs.append(update_error_adafactor(grad, v, v_q) * 100)
        mn = sum(errs) / len(errs)
        sd = (sum((e - mn) ** 2 for e in errs) / len(errs)) ** 0.5
        print(f"  {ml:<8} {mn:>8.3f} {sd:>8.3f} {min(errs):>8.3f} {max(errs):>8.3f}")
    # adaptive reference
    errs = []
    for trial in range(n_trials):
        v = gen_v(dname, 8192, seed=trial * 100 + 7)
        rng = torch.Generator().manual_seed(trial * 100 + 8)
        grad = torch.randn(8192, generator=rng) * 0.01
        v_q = qd_log_adaptive(v, bs)
        errs.append(update_error_adafactor(grad, v, v_q) * 100)
    mn = sum(errs) / len(errs)
    sd = (sum((e - mn) ** 2 for e in errs) / len(errs)) ** 0.5
    print(f"  {'adapt':<8} {mn:>8.3f} {sd:>8.3f} {min(errs):>8.3f} {max(errs):>8.3f}")

A2: Stability (10 seeds, L2 update error %)

  narrow_0.5dec
  min_log      mean%      std%      min%      max%
  ------------------------------------------------
  -126        4.352    0.071    4.286    4.517
  -53         1.578    0.014    1.557    1.596
  -40         1.064    0.013    1.042    1.082
  -32         0.754    0.005    0.742    0.762
  -24         0.439    0.004    0.434    0.447
  adapt       0.041    0.001    0.038    0.044

  medium_10dec
  min_log      mean%      std%      min%      max%
  ------------------------------------------------
  -126        4.701    0.186    4.380    5.160
  -53         1.939    0.098    1.755    2.070
  -40         1.408    0.131    1.237    1.698
  -32         1.066    0.059    0.965    1.159
  -24        15.536    6.045    8.810   29.760
  adapt       0.799    0.037    0.755    0.866

  wide_20dec
  min_log      mean%      std%      min%      max%
  ------------------------------------------------
  -126        4.730    0.909    3.124  

In [5]:
# A3: EMA drift with fixed min_log (5000 steps)
# Shows that fixed min_log error accumulates over time, while adaptive stays low.

n = 8192
bs = 2048
beta_val = 0.001
steps = 5000

rng = torch.Generator().manual_seed(123)
grads = [torch.randn(n, generator=rng) * 0.01 for _ in range(steps)]

v_ref = torch.zeros(n)
for g in grads:
    v_ref = (1 - beta_val) * v_ref + beta_val * g.square()

configs = {
    "adaptive":     lambda v: qd_log_adaptive(v, bs),
    "fixed -126":   lambda v: qd_log_fixed(v, bs, -126.0),
    "fixed -53":    lambda v: qd_log_fixed(v, bs, -53.0),
    "fixed -40":    lambda v: qd_log_fixed(v, bs, -40.0),
    "fixed -28":    lambda v: qd_log_fixed(v, bs, -28.0),
    "bnb":          lambda v: qd_bnb(v, bs),
}

print(f"A3: EMA drift ({steps} steps, beta={beta_val}, bs={bs})")
print(f"  {'method':<14} {'drift%':>10}")
print("  " + "-" * 28)

for cname, qfn in configs.items():
    v_stored = torch.zeros(n)
    for g in grads:
        v_new = (1 - beta_val) * v_stored + beta_val * g.square()
        v_stored = qfn(v_new)
    drift = rel_l2(v_ref, v_stored) * 100
    print(f"  {cname:<14} {drift:>9.3f}")

A3: EMA drift (5000 steps, beta=0.001, bs=2048)
  method             drift%
  ----------------------------
  adaptive           0.736
  fixed -126        56.914
  fixed -53         52.956
  fixed -40         74.463
  fixed -28         25.254
  bnb               10.032


In [6]:
# A4: Cold embedding analysis
# Sparse V: most elements are zero (cold tokens), few are active (hot tokens).
# n=4096 and grad_std=0.05 reflect embedding-layer scale.

n = 4096
bs = 2048
sparsities = [0.4, 0.5, 0.9, 0.99]
min_logs_test = [-126, -53, -40, -28]

print("A4: Cold embedding analysis (single-step)")
print("  L1 on non-zero elements, L2 with kernel clamp")

for sp in sparsities:
    v = simulate_v_ema(0.05, 0.001, 500, n, sparsity=sp, seed=42)
    rng = torch.Generator().manual_seed(7)
    grad = torch.randn(n, generator=rng) * 0.05
    n_zero = (v == 0).sum().item()
    n_hot = n - n_zero
    v_nz = v[v > 0]
    v_range = f"[{v_nz.min():.1e}, {v_nz.max():.1e}]" if n_hot > 0 else "N/A"

    print(f"\n  sparsity={sp:.0%} (zeros={n_zero}, hot={n_hot}, hot V range={v_range})")
    print(f"  {'method':<18} {'L1_qnt%':>9} {'L2_upd%':>9}")
    print("  " + "-" * 40)

    for mname, mfn in [
        ("adaptive 1+255", lambda v: qd_log_adaptive(v, bs)),
        ("adaptive 0+255", lambda v: qd_log_adaptive_nz(v, bs)),
    ]:
        v_q = mfn(v)
        l1 = rel_l2(v, v_q) * 100
        l2 = update_error_adafactor(grad, v, v_q) * 100
        print(f"  {mname:<18} {l1:>8.3f} {fmt_err(l2)}")

    for ml in min_logs_test:
        v_q = qd_log_fixed(v, bs, float(ml))
        l1 = rel_l2(v, v_q) * 100
        l2 = update_error_adafactor(grad, v, v_q) * 100
        print(f"  {'fixed '+str(ml):<18} {l1:>8.3f} {fmt_err(l2)}")

A4: Cold embedding analysis (single-step)
  L1 on non-zero elements, L2 with kernel clamp

  sparsity=40% (zeros=1671, hot=2425, hot V range=[8.1e-04, 1.2e-03])
  method               L1_qnt%   L2_upd%
  ----------------------------------------
  adaptive 1+255        0.042     0.000
  adaptive 0+255       11.236     0.000
  fixed -126           11.236     0.000
  fixed -53             3.379     >100%
  fixed -40             2.376     >100%
  fixed -28             1.427     >100%

  sparsity=50% (zeros=2051, hot=2045, hot V range=[8.1e-04, 1.2e-03])
  method               L1_qnt%   L2_upd%
  ----------------------------------------
  adaptive 1+255        0.042     0.000
  adaptive 0+255       11.252     0.000
  fixed -126           11.252     0.000
  fixed -53             3.385     >100%
  fixed -40             2.379     >100%
  fixed -28             1.431     >100%

  sparsity=90% (zeros=3698, hot=398, hot V range=[8.2e-04, 1.2e-03])
  method               L1_qnt%   L2_upd%
  -------

## §B Head-to-Head Comparison

log_adapt (ours) vs bnb (baseline). 1D uses Adam-style metric, 2D uses Adafactor-style.

In [7]:
# B1: Single-step V quantization — ours vs bnb
# 1D: Adam-style metric (eps=1e-8, standard Adam default; kernel default eps1=1e-30
#     does not affect relative ranking between methods).
# 2D factored: Adafactor-style metric (rsqrt).

bs_list = [256, 2048]
v_methods = {
    "log_adapt": qd_log_adaptive,
    "bnb":       qd_bnb,
}

# --- 1D distributions ---
dists_1d = ["narrow_0.5dec", "narrow_3dec", "medium_10dec", "wide_20dec",
            "outlier", "bimodal", "ema_realistic", "sparse_90pct"]

print("B1a: 1D V quantization (Adam-style: 1/(sqrt(v)+eps), eps=1e-8)")
print(f"  {'distribution':<18}", end="")
for bs in bs_list:
    for mn in v_methods:
        print(f" {mn+'_'+str(bs):>16}", end="")
print()
print("  " + "-" * (18 + 17 * len(bs_list) * len(v_methods)))

for dname in dists_1d:
    v = gen_v(dname, 8192, seed=42)
    rng = torch.Generator().manual_seed(7)
    grad = torch.randn(8192, generator=rng) * 0.01
    print(f"  {dname:<18}", end="")
    for bs in bs_list:
        for mn, mfn in v_methods.items():
            v_q = mfn(v, bs)
            ue = update_error_adam(grad, v, v_q, eps=1e-8) * 100
            print(f" {fmt_err(ue):>16}", end="")
    print()

# --- 2D factored distributions ---
print(f"\nB1b: 2D factored V quantization (Adafactor-style: rsqrt)")
print(f"  {'distribution':<18}", end="")
for bs in bs_list:
    for mn in v_methods:
        print(f" {mn+'_'+str(bs):>16}", end="")
print()
print("  " + "-" * (18 + 17 * len(bs_list) * len(v_methods)))

for dname in ["ada_dense", "ada_mixed", "ada_cold"]:
    v = gen_ada_v(dname, R=64, C=128, seed=42)
    rng = torch.Generator().manual_seed(7)
    grad = torch.randn_like(v) * 0.01
    print(f"  {dname:<18}", end="")
    for bs in bs_list:
        for mn, mfn in v_methods.items():
            v_q = mfn(v, bs)
            ue = update_error_adafactor(grad, v, v_q) * 100
            print(f" {fmt_err(ue):>16}", end="")
    print()

B1a: 1D V quantization (Adam-style: 1/(sqrt(v)+eps), eps=1e-8)
  distribution          log_adapt_256          bnb_256   log_adapt_2048         bnb_2048
  --------------------------------------------------------------------------------------
  narrow_0.5dec                 0.033            0.136            0.039            0.143
  narrow_3dec                   0.215            0.545            0.260            0.582
  medium_10dec                  0.530            6.427            0.688            9.249
  wide_20dec                    0.386            >100%            0.710            >100%
  outlier                       0.129            0.592            0.301            2.202
  bimodal                       0.182            >100%            0.256            >100%
  ema_realistic                 0.021            0.121            0.025            0.124
  sparse_90pct                  0.000            0.000            0.000            0.000

B1b: 2D factored V quantization (Adafactor-sty

In [8]:
# B2: Multi-step EMA drift (5000 steps)
# M is fp32 throughout; isolates V quantization effect.
# B2a: 1D element-wise (Adam-style). B2b: 2D factored (Adafactor-style, corrected feedback).
# Update error is measured at the final step (steady-state), not per-step average.

steps = 5000
n = 8192

# --- B2a: 1D EMA drift ---
beta1, beta2 = 0.9, 0.999
beta_val = 1.0 - beta2
eps = 1e-8

rng = torch.Generator().manual_seed(123)
grads = [torch.randn(n, generator=rng) * 0.01 for _ in range(steps)]

m_ref = torch.zeros(n)
v_ref = torch.zeros(n)
for g in grads:
    m_ref = beta1 * m_ref + (1 - beta1) * g
    v_ref = beta2 * v_ref + beta_val * g.square()
bc2 = 1.0 - beta2 ** steps
u_ref = m_ref / (v_ref.sqrt() + eps * math.sqrt(bc2))

configs_1d = {
    "log_adapt_256":  lambda v: qd_log_adaptive(v, 256),
    "log_adapt_2048": lambda v: qd_log_adaptive(v, 2048),
    "bnb_256":        lambda v: qd_bnb(v, 256),
    "bnb_2048":       lambda v: qd_bnb(v, 2048),
}

print(f"B2a: 1D EMA drift ({steps} steps, beta1={beta1}, beta2={beta2})")
print(f"  (M is fp32; steady-state update error at step {steps})")
print(f"  {'method':<18} {'V_drift%':>10} {'upd_err%':>10}")
print("  " + "-" * 42)

for cname, qfn in configs_1d.items():
    m_q = torch.zeros(n)
    v_stored = torch.zeros(n)
    for t, g in enumerate(grads):
        v_new = beta2 * v_stored + beta_val * g.square()
        m_q = beta1 * m_q + (1 - beta1) * g
        v_stored = qfn(v_new)
    bc2_t = 1.0 - beta2 ** steps
    u_q = m_q / (v_stored.sqrt() + eps * math.sqrt(bc2_t))
    v_drift = rel_l2(v_ref, v_stored) * 100
    u_err = ((u_q - u_ref).norm() / u_ref.norm()).item() * 100
    print(f"  {cname:<18} {v_drift:>9.3f} {fmt_err(u_err)}")

# --- B2b: 2D factored EMA drift (quantize row/col, not reconstructed V) ---
R, C = 64, 128
beta2_ada = 0.999
beta_val_ada = 1.0 - beta2_ada
eps1 = 1e-30

rng2 = torch.Generator().manual_seed(456)
grads_2d = [torch.randn(R, C, generator=rng2) * 0.01 for _ in range(steps)]

row_ref = torch.zeros(R)
col_ref = torch.zeros(C)
for g in grads_2d:
    g_sq = g.square()
    row_ref = beta2_ada * row_ref + beta_val_ada * (g_sq.mean(dim=1) + eps1)
    col_ref = beta2_ada * col_ref + beta_val_ada * (g_sq.mean(dim=0) + eps1)
V_ref = (row_ref.unsqueeze(1) * col_ref.unsqueeze(0) / row_ref.mean().clamp(min=1e-30)).flatten()

configs_2d = {
    "log_adapt_256":  lambda v: qd_log_adaptive(v, 256),
    "log_adapt_2048": lambda v: qd_log_adaptive(v, 2048),
    "bnb_256":        lambda v: qd_bnb(v, 256),
    "bnb_2048":       lambda v: qd_bnb(v, 2048),
}

print(f"\nB2b: 2D factored EMA drift ({steps} steps, R={R}, C={C})")
print(f"  (M is fp32; quantize row/col statistics separately, kernel-aligned)")
print(f"  {'method':<18} {'row_drift%':>11} {'col_drift%':>11} {'V_drift%':>10}")
print("  " + "-" * 54)

for cname, qfn in configs_2d.items():
    row_stored = torch.zeros(R)
    col_stored = torch.zeros(C)
    for g in grads_2d:
        g_sq = g.square()
        row_new = beta2_ada * row_stored + beta_val_ada * (g_sq.mean(dim=1) + eps1)
        col_new = beta2_ada * col_stored + beta_val_ada * (g_sq.mean(dim=0) + eps1)
        row_stored = qfn(row_new)
        col_stored = qfn(col_new)
    V_q = (row_stored.unsqueeze(1) * col_stored.unsqueeze(0)
           / row_stored.mean().clamp(min=1e-30)).flatten()
    row_drift = rel_l2(row_ref, row_stored) * 100
    col_drift = rel_l2(col_ref, col_stored) * 100
    v_drift = rel_l2(V_ref, V_q) * 100
    print(f"  {cname:<18} {row_drift:>10.3f} {col_drift:>10.3f} {v_drift:>9.3f}")

B2a: 1D EMA drift (5000 steps, beta1=0.9, beta2=0.999)
  (M is fp32; steady-state update error at step 5000)
  method               V_drift%   upd_err%
  ------------------------------------------
  log_adapt_256          0.562     0.280
  log_adapt_2048         0.736     0.357
  bnb_256                7.282     3.436
  bnb_2048              10.032     4.577

B2b: 2D factored EMA drift (5000 steps, R=64, C=128)
  (M is fp32; quantize row/col statistics separately, kernel-aligned)
  method              row_drift%  col_drift%   V_drift%
  ------------------------------------------------------
  log_adapt_256           0.040      0.054     0.067
  log_adapt_2048          0.040      0.054     0.067
  bnb_256                 3.836      5.087     5.385
  bnb_2048                3.836      5.087     5.385


In [9]:
# B2b: 2D factored EMA drift (quantize row/col statistics, kernel-aligned)
# M is fp32 throughout; isolates V quantization effect.

eps1 = 1e-30

def run_2d_drift(R, C, steps, seed, configs, row_scale=None):
    beta2_ada = 0.999
    bv = 1.0 - beta2_ada
    if row_scale is None:
        row_scale = torch.ones(R)

    rng_ref = torch.Generator().manual_seed(seed)
    row_ref = torch.zeros(R)
    col_ref = torch.zeros(C)
    for _ in range(steps):
        g = torch.randn(R, C, generator=rng_ref) * 0.01 * row_scale.unsqueeze(1)
        g_sq = g.square()
        row_ref = beta2_ada * row_ref + bv * (g_sq.mean(dim=1) + eps1)
        col_ref = beta2_ada * col_ref + bv * (g_sq.mean(dim=0) + eps1)
    V_ref = (row_ref.unsqueeze(1) * col_ref.unsqueeze(0)
             / row_ref.mean().clamp(min=1e-30)).flatten()

    results = {}
    for cname, qfn in configs.items():
        rng_q = torch.Generator().manual_seed(seed)
        row_stored = torch.zeros(R)
        col_stored = torch.zeros(C)
        for _ in range(steps):
            g = torch.randn(R, C, generator=rng_q) * 0.01 * row_scale.unsqueeze(1)
            g_sq = g.square()
            row_new = beta2_ada * row_stored + bv * (g_sq.mean(dim=1) + eps1)
            col_new = beta2_ada * col_stored + bv * (g_sq.mean(dim=0) + eps1)
            row_stored = qfn(row_new)
            col_stored = qfn(col_new)
        V_q = (row_stored.unsqueeze(1) * col_stored.unsqueeze(0)
               / row_stored.mean().clamp(min=1e-30)).flatten()
        results[cname] = (
            rel_l2(row_ref, row_stored) * 100,
            rel_l2(col_ref, col_stored) * 100,
            rel_l2(V_ref, V_q) * 100,
        )
    return results

def print_2d_results(title, note, results, configs):
    print(f"{title}")
    print(f"  {note}")
    print(f"  {'method':<18} {'row_drift%':>11} {'col_drift%':>11} {'V_drift%':>10}")
    print("  " + "-" * 54)
    for cname in configs:
        rd, cd, vd = results[cname]
        print(f"  {cname:<18} {rd:>10.3f} {cd:>10.3f} {vd:>9.3f}")

configs_2d = {
    "log_adapt_256":  lambda v: qd_log_adaptive(v, 256),
    "log_adapt_2048": lambda v: qd_log_adaptive(v, 2048),
    "bnb_256":        lambda v: qd_bnb(v, 256),
    "bnb_2048":       lambda v: qd_bnb(v, 2048),
}

# --- Small layer: row/col fit in one block ---
res = run_2d_drift(64, 128, 5000, 456, configs_2d)
print_2d_results("B2b-small: R=64, C=128, 5000 steps",
                 "(row/col < block_size, single block)", res, configs_2d)

# --- Large layer, homogeneous rows ---
print()
res = run_2d_drift(512, 512, 2000, 789, configs_2d)
print_2d_results("B2b-large: R=512, C=512, 2000 steps, homogeneous rows",
                 "(row/col span 2 blocks at bs=256)", res, configs_2d)

# --- Large layer, heterogeneous rows (hot/normal/cold) ---
hetero_scale = torch.ones(512)
hetero_scale[:128] = 10.0     # hot: grad ~0.1, row_var ~1e-2
hetero_scale[384:] = 0.01     # cold: grad ~1e-4, row_var ~1e-8
# row_var spans ~6 orders of magnitude across blocks

print()
res = run_2d_drift(512, 512, 2000, 789, configs_2d, row_scale=hetero_scale)
print_2d_results("B2b-hetero: R=512, C=512, 2000 steps, heterogeneous rows",
                 "(row_var spans ~6 decades: hot 1e-2, normal 1e-4, cold 1e-8)",
                 res, configs_2d)

B2b-small: R=64, C=128, 5000 steps
  (row/col < block_size, single block)
  method              row_drift%  col_drift%   V_drift%
  ------------------------------------------------------
  log_adapt_256           0.040      0.054     0.067
  log_adapt_2048          0.040      0.054     0.067
  bnb_256                 3.836      5.087     5.385
  bnb_2048                3.836      5.087     5.385

B2b-large: R=512, C=512, 2000 steps, homogeneous rows
  (row/col span 2 blocks at bs=256)
  method              row_drift%  col_drift%   V_drift%
  ------------------------------------------------------
  log_adapt_256           0.023      0.022     0.032
  log_adapt_2048          0.025      0.021     0.032
  bnb_256                 1.312      1.210     1.777
  bnb_2048                3.140      3.277     3.491

B2b-hetero: R=512, C=512, 2000 steps, heterogeneous rows
  (row_var spans ~6 decades: hot 1e-2, normal 1e-4, cold 1e-8)
  method              row_drift%  col_drift%   V_drift%
  ------

In [10]:
# B3: Win/Loss matrix — log_adapt_256 vs bnb_256 (30 trials)
# Win = >1% lower update error. Metric: Adam-style (eps=1e-8).

n = 8192
n_trials = 30
dists = ["narrow_0.5dec", "narrow_3dec", "medium_10dec", "wide_20dec",
         "outlier", "bimodal", "ema_realistic", "sparse_90pct"]

print(f"B3: Win/Loss — log_adapt_256 vs bnb_256 ({n_trials} trials, >1% threshold)")
print(f"  {'distribution':<18} {'log_adapt W/L':>14} {'mean_ours%':>11} {'mean_bnb%':>11}")
print("  " + "-" * 58)

total_w, total_l, total_n = 0, 0, 0
for di, dname in enumerate(dists):
    w, l = 0, 0
    errs_ours, errs_bnb = [], []
    for trial in range(n_trials):
        seed = trial * 1000 + di * 137
        v = gen_v(dname, n, seed=seed)
        rng = torch.Generator().manual_seed(seed + 1)
        grad = torch.randn(n, generator=rng) * 0.01
        e_ours = update_error_adam(grad, v, qd_log_adaptive(v, 256), eps=1e-8)
        e_bnb = update_error_adam(grad, v, qd_bnb(v, 256), eps=1e-8)
        errs_ours.append(e_ours)
        errs_bnb.append(e_bnb)
        if e_ours < e_bnb * 0.99:
            w += 1
        elif e_ours > e_bnb * 1.01:
            l += 1
    total_w += w; total_l += l; total_n += n_trials
    mo = sum(errs_ours) / len(errs_ours) * 100
    mb = sum(errs_bnb) / len(errs_bnb) * 100
    print(f"  {dname:<18} {w:>5}W /{l:>3}L {fmt_err(mo)} {fmt_err(mb)}")

print(f"\n  TOTAL: {total_w}W / {total_l}L out of {total_n}")

B3: Win/Loss — log_adapt_256 vs bnb_256 (30 trials, >1% threshold)
  distribution        log_adapt W/L  mean_ours%   mean_bnb%
  ----------------------------------------------------------
  narrow_0.5dec         30W /  0L     0.034     0.139
  narrow_3dec           30W /  0L     0.222     0.570
  medium_10dec          30W /  0L     0.568     >100%
  wide_20dec            30W /  0L     0.773     >100%
  outlier               30W /  0L     0.137     0.728
  bimodal               30W /  0L     0.172     >100%
  ema_realistic         30W /  0L     0.021     0.122
  sparse_90pct          30W /  0L     0.000     0.000

  TOTAL: 240W / 0L out of 240


In [11]:
# B4: Stability across 10 seeds (update error %)
# Metric: Adam-style.

n = 8192
n_trials = 10
dists = ["narrow_0.5dec", "narrow_3dec", "medium_10dec", "wide_20dec",
         "outlier", "bimodal"]

methods = {
    "log_adapt_256":  lambda v: qd_log_adaptive(v, 256),
    "log_adapt_2048": lambda v: qd_log_adaptive(v, 2048),
    "bnb_256":        lambda v: qd_bnb(v, 256),
    "bnb_2048":       lambda v: qd_bnb(v, 2048),
}

print(f"B4: Stability ({n_trials} seeds, update error %)")

for dname in dists:
    print(f"\n  {dname}")
    print(f"  {'method':<18} {'mean%':>9} {'std%':>9} {'min%':>9} {'max%':>9}")
    print("  " + "-" * 58)
    for mname, mfn in methods.items():
        errs = []
        for trial in range(n_trials):
            v = gen_v(dname, n, seed=trial * 100 + 7)
            rng = torch.Generator().manual_seed(trial * 100 + 8)
            grad = torch.randn(n, generator=rng) * 0.01
            errs.append(update_error_adam(grad, v, mfn(v), eps=1e-8))
        mn = sum(errs) / len(errs) * 100
        sd = (sum((e * 100 - mn) ** 2 for e in errs) / len(errs)) ** 0.5
        lo = min(errs) * 100
        hi = max(errs) * 100
        print(f"  {mname:<18} {mn:>8.3f} {sd:>8.3f} {lo:>8.3f} {hi:>8.3f}")

B4: Stability (10 seeds, update error %)

  narrow_0.5dec
  method                 mean%      std%      min%      max%
  ----------------------------------------------------------
  log_adapt_256         0.034    0.001    0.032    0.035
  log_adapt_2048        0.041    0.001    0.038    0.044
  bnb_256               0.140    0.001    0.137    0.142
  bnb_2048              0.149    0.002    0.145    0.153

  narrow_3dec
  method                 mean%      std%      min%      max%
  ----------------------------------------------------------
  log_adapt_256         0.223    0.004    0.215    0.230
  log_adapt_2048        0.274    0.008    0.258    0.292
  bnb_256               0.576    0.023    0.549    0.620
  bnb_2048              0.649    0.041    0.607    0.753

  medium_10dec
  method                 mean%      std%      min%      max%
  ----------------------------------------------------------
  log_adapt_256         0.587    0.026    0.544    0.630
  log_adapt_2048        0.799   

In [12]:
# B5: Path generalization — V comparison across optimizer paths
# Isolates V effect by using fp32 M. Tests whether log_adapt's advantage
# holds in CAME's residual/confidence mechanism, not just Adafactor.

R, C = 64, 128
steps = 10000
beta1, beta2, beta3 = 0.9, 0.999, 0.9999
eps1, eps_came = 1e-30, 1e-16
clip_threshold = 1.0

def asg(row, col):
    return (row / row.mean().clamp(min=eps1)).rsqrt().unsqueeze(-1) * col.unsqueeze(-2).rsqrt()

rng = torch.Generator().manual_seed(42)
grads = [torch.randn(R, C, generator=rng) * 0.01 for _ in range(steps)]

v_methods = {"log_adapt": lambda v: qd_log_adaptive(v, 2048),
             "bnb":       lambda v: qd_bnb(v, 256)}

def run_path_ref(path):
    row = torch.zeros(R); col = torch.zeros(C)
    m = torch.zeros(R, C); rr = torch.zeros(R); rc = torch.zeros(C)
    refs = []
    for g in grads:
        g_sq = g.square() + eps1
        row = beta2 * row + (1 - beta2) * g_sq.mean(dim=-1)
        col = beta2 * col + (1 - beta2) * g_sq.mean(dim=-2)
        u = asg(row, col) * g
        rms = u.norm() / math.sqrt(u.numel())
        u = u / max((rms / clip_threshold).item(), 1.0)
        if path == "ada_nom":
            refs.append(u.clone())
        elif path == "ada_m":
            m = beta1 * m + (1 - beta1) * u
            refs.append(m.clone())
        else:
            m = beta1 * m + (1 - beta1) * u
            res = (u - m).square() + eps_came
            rr = beta3 * rr + (1 - beta3) * res.mean(dim=-1)
            rc = beta3 * rc + (1 - beta3) * res.mean(dim=-2)
            refs.append((asg(rr, rc) * m).clone())
    return refs

def run_path_quant(path, qv):
    row = torch.zeros(R); col = torch.zeros(C)
    m = torch.zeros(R, C); rr = torch.zeros(R); rc = torch.zeros(C)
    errs = []
    for t, g in enumerate(grads):
        g_sq = g.square() + eps1
        row = qv(beta2 * row + (1 - beta2) * g_sq.mean(dim=-1))
        col = qv(beta2 * col + (1 - beta2) * g_sq.mean(dim=-2))
        u = asg(row, col) * g
        rms = u.norm() / math.sqrt(u.numel())
        u = u / max((rms / clip_threshold).item(), 1.0)
        if path == "ada_nom":
            final = u
        elif path == "ada_m":
            m = beta1 * m + (1 - beta1) * u
            final = m
        else:
            m = beta1 * m + (1 - beta1) * u
            res = (u - m).square() + eps_came
            rr = qv(beta3 * rr + (1 - beta3) * res.mean(dim=-1))
            rc = qv(beta3 * rc + (1 - beta3) * res.mean(dim=-2))
            final = asg(rr, rc) * m
        ref_n = refs[path][t].norm()
        errs.append(((final - refs[path][t]).norm() / ref_n).item() if ref_n > 1e-30 else 0.0)
    return errs

print("Precomputing fp32 references...")
refs = {p: run_path_ref(p) for p in ["ada_nom", "ada_m", "came"]}

paths = [("Ada(no-M)", "ada_nom"), ("Ada(+M, fp32)", "ada_m"), ("CAME (fp32 M)", "came")]
windows = [(0, 1000), (1000, 5000), (5000, 10000)]

print(f"\nB5: V path generalization ({steps} steps, M=fp32, R={R}, C={C})")
print(f"\n  {'Path':<18} {'V':<12}", end="")
for s, e in windows:
    print(f" {s//1000}k-{e//1000}k%", end="")
print()
print("  " + "-" * (18 + 12 + 10 * len(windows)))

for pname, pkey in paths:
    for vname, qv in v_methods.items():
        errs = run_path_quant(pkey, qv)
        means = [sum(errs[s:e]) / (e - s) * 100 for s, e in windows]
        print(f"  {pname:<18} {vname:<12}", end="")
        for m in means:
            print(f" {m:>9.4f}", end="")
        print()

Precomputing fp32 references...

B5: V path generalization (10000 steps, M=fp32, R=64, C=128)

  Path               V            0k-1k% 1k-5k% 5k-10k%
  ------------------------------------------------------------
  Ada(no-M)          log_adapt       0.0386    0.0322    0.0304
  Ada(no-M)          bnb             1.0953    1.2404    1.3201
  Ada(+M, fp32)      log_adapt       0.0387    0.0323    0.0304
  Ada(+M, fp32)      bnb             1.0915    1.2409    1.3299
  CAME (fp32 M)      log_adapt       0.0487    0.0422    0.0361
  CAME (fp32 M)      bnb             1.7889    2.5887    3.0110


## §C Theoretical Properties

C1/C3/C5 use per-block quantization (bs=2048 unless noted).
C2/C4 use global quantization (theoretical analysis).

In [13]:
# C1: V bias accumulation (5000 steps, per-block bs=2048)
# bias = mean(V_quant / V_fp32 - 1) on non-zero elements

n = 8192
beta_val = 0.001
steps = 50000
bs = 2048
report_every = 2000

rng = torch.Generator().manual_seed(123)
grads = [torch.randn(n, generator=rng) * 0.01 for _ in range(steps)]

v_ref = torch.zeros(n)
configs = {
    "log_adapt": lambda v: qd_log_adaptive(v, bs),
    "bnb":       lambda v: qd_bnb(v, bs),
}
v_states = {name: torch.zeros(n) for name in configs}

print(f"C1: V bias accumulation ({steps} steps, per-block bs={bs})")
print(f"\n  {'step':<8}", end="")
for name in configs:
    print(f" {name:>12}", end="")
print()
print("  " + "-" * (8 + 13 * len(configs)))

for t in range(steps):
    g_sq = grads[t].square()
    v_ref = (1 - beta_val) * v_ref + beta_val * g_sq
    for name, qfn in configs.items():
        v_states[name] = (1 - beta_val) * v_states[name] + beta_val * g_sq
        v_states[name] = qfn(v_states[name])
    if (t + 1) % report_every == 0:
        mask = v_ref > 0
        print(f"  {t+1:<8}", end="")
        for name in configs:
            bias = ((v_states[name][mask] / v_ref[mask]) - 1.0).mean().item()
            print(f" {bias*100:>+11.4f}%", end="")
        print()

C1: V bias accumulation (50000 steps, per-block bs=2048)

  step        log_adapt          bnb
  ----------------------------------
  2000         +0.4328%     +1.7680%
  4000         +0.4947%     +2.7815%
  6000         +0.4709%    +10.6854%
  8000         +0.5124%    +13.8411%
  10000        +0.5411%     +9.0519%
  12000        +0.5718%     +8.3803%
  14000        +0.5430%    +11.0546%
  16000        +0.5073%     +9.8111%
  18000        +0.4640%     +7.7478%
  20000        +0.5602%     +9.6119%
  22000        +0.5544%     +7.9363%
  24000        +0.5474%    +12.2812%
  26000        +0.5539%     +8.8210%
  28000        +0.5485%     +9.8859%
  30000        +0.5239%     +8.3001%
  32000        +0.5305%     +3.7336%
  34000        +0.5531%     +5.7385%
  36000        +0.5283%    +10.6778%
  38000        +0.6127%    +10.1778%
  40000        +0.5049%     +8.3771%
  42000        +0.5149%     +8.1202%
  44000        +0.5056%     +4.3081%
  46000        +0.4827%     +9.1376%
  48000        +0

In [14]:
# C2: Pure Jensen bias (single roundtrip, global quantization)
# Measures E[V_deq / V - 1] for one quantize-dequantize cycle.

n_large = 100000
n_trials = 20

c2_dists = {
    "narrow (0.5 dec)": lambda rng: torch.pow(2.0, torch.randn(n_large, generator=rng) * 0.15 - 13.3),
    "medium (3 dec)":   lambda rng: torch.pow(2.0, torch.randn(n_large, generator=rng) * 1.0 - 13.3),
    "wide (10 dec)":    lambda rng: torch.pow(2.0, torch.randn(n_large, generator=rng) * 3.0 - 15.0),
}
c2_methods = {
    "log_adapt": lambda v: qd_log_adaptive(v, v.numel()),
    "bnb":       lambda v: qd_bnb(v, v.numel()),
}

print("C2: Pure Jensen bias (single roundtrip, global quantization)")
print(f"\n  {'distribution':<22}", end="")
for name in c2_methods:
    print(f" {name:>12}", end="")
print()
print("  " + "-" * (22 + 13 * len(c2_methods)))

for dname, dgen in c2_dists.items():
    print(f"  {dname:<22}", end="")
    for mname, mfn in c2_methods.items():
        biases = []
        for trial in range(n_trials):
            rng = torch.Generator().manual_seed(trial)
            v = dgen(rng)
            v_deq = mfn(v)
            biases.append(((v_deq / v) - 1.0).mean().item())
        mean_bias = sum(biases) / len(biases)
        print(f" {mean_bias*100:>+11.4f}%", end="")
    print()

C2: Pure Jensen bias (single roundtrip, global quantization)

  distribution              log_adapt          bnb
  ------------------------------------------------
  narrow (0.5 dec)           -0.0001%     -0.0000%
  medium (3 dec)             +0.0016%     -0.0099%
  wide (10 dec)              +0.0194%     -0.9256%


In [15]:
# C3: Per-step update error across training (5000 steps, per-block)
# Compares log_adapt at two block sizes vs bnb.

n = 8192
beta1, beta2 = 0.9, 0.999
beta_val = 1.0 - beta2
eps = 1e-8
steps = 5000

rng = torch.Generator().manual_seed(789)
grads = [torch.randn(n, generator=rng) * 0.01 for _ in range(steps)]

v_ref = torch.zeros(n)
m_ref = torch.zeros(n)
u_refs = []
for g in grads:
    v_ref = beta2 * v_ref + beta_val * g.square()
    m_ref = beta1 * m_ref + (1 - beta1) * g
    u_refs.append(m_ref / (v_ref.sqrt() + eps))

configs = {
    "log_adapt_256":  lambda v: qd_log_adaptive(v, 256),
    "log_adapt_2048": lambda v: qd_log_adaptive(v, 2048),
    "bnb_256":        lambda v: qd_bnb(v, 256),
}

errors = {name: [] for name in configs}
v_stored = {name: torch.zeros(n) for name in configs}
m_q = {name: torch.zeros(n) for name in configs}

for t in range(steps):
    g = grads[t]
    for name, qfn in configs.items():
        v_new = beta2 * v_stored[name] + beta_val * g.square()
        m_q[name] = beta1 * m_q[name] + (1 - beta1) * g
        u_q = m_q[name] / (v_new.sqrt() + eps)
        v_stored[name] = qfn(v_new)
        ref_n = u_refs[t].norm()
        errors[name].append(((u_q - u_refs[t]).norm() / ref_n).item() if ref_n > 1e-30 else 0.0)

windows = [(0, 100), (100, 500), (500, 1000), (1000, 2000), (2000, 3000), (3000, 5000)]
print(f"C3: Per-step update error ({steps} steps, per-block)")
print(f"\n  {'window':<14}", end="")
for name in configs:
    print(f" {name:>16}", end="")
print()
print("  " + "-" * (14 + 17 * len(configs)))

for start, end in windows:
    print(f"  {start:>5}-{end:<5}  ", end="")
    for name in configs:
        mean_e = sum(errors[name][start:end]) / (end - start)
        print(f" {mean_e*100:>15.4f}%", end="")
    print()

C3: Per-step update error (5000 steps, per-block)

  window            log_adapt_256   log_adapt_2048          bnb_256
  -----------------------------------------------------------------
      0-100              0.3287%          0.3997%          0.7657%
    100-500              0.3064%          0.3706%          1.1912%
    500-1000             0.3005%          0.3727%          1.5624%
   1000-2000             0.2969%          0.3600%          2.0997%
   2000-3000             0.2905%          0.3529%          2.5407%
   3000-5000             0.2854%          0.3599%          3.0088%


In [16]:
# C4: Adaptive range stability (min_log/max_log drift per step)
# Global quantization (theoretical analysis of grid convergence).

n = 8192
beta_val = 0.001
steps = 3000

rng = torch.Generator().manual_seed(101)
grads = [torch.randn(n, generator=rng) * 0.01 for _ in range(steps)]

v = torch.zeros(n)
prev_min_log = None
prev_max_log = None
min_log_drifts = []
max_log_drifts = []
ranges = []

for t in range(steps):
    v = (1 - beta_val) * v + beta_val * grads[t].square()
    v_nz = v[v > 0]
    if v_nz.numel() > 0:
        cur_min = torch.log2(v_nz.min()).item()
        cur_max = torch.log2(v_nz.max()).item()
    else:
        cur_min, cur_max = 0.0, 0.0
    if prev_min_log is not None:
        min_log_drifts.append(abs(cur_min - prev_min_log))
        max_log_drifts.append(abs(cur_max - prev_max_log))
    ranges.append(cur_max - cur_min)
    prev_min_log, prev_max_log = cur_min, cur_max

windows = [(0, 50), (50, 200), (200, 500), (500, 1000), (1000, 2000), (2000, 3000)]
print(f"C4: Adaptive range stability ({steps} steps)")
print(f"\n  {'window':<14} {'d_min_log':>10} {'d_max_log':>10} {'range':>10}")
print("  " + "-" * 48)
for start, end in windows:
    e = min(end, len(min_log_drifts))
    if start < e:
        dm = sum(min_log_drifts[start:e]) / (e - start)
        dM = sum(max_log_drifts[start:e]) / (e - start)
        r = sum(ranges[start:e+1]) / (e - start + 1)
        print(f"  {start:>5}-{end:<5}  {dm:>9.4f} {dM:>9.4f} {r:>9.2f}")

C4: Adaptive range stability (3000 steps)

  window          d_min_log  d_max_log      range
  ------------------------------------------------
      0-50        0.5656    0.0459      4.61
     50-200       0.0174    0.0116      1.47
    200-500       0.0050    0.0038      0.88
    500-1000      0.0026    0.0022      0.61
   1000-2000      0.0018    0.0016      0.43
   2000-3000      0.0015    0.0014      0.38


In [17]:
# C5: Parameter drift (full Adam, per-block)
# fp32 M rows isolate V contribution. UF8/D8 rows show practical configurations.

n = 8192
beta1, beta2 = 0.9, 0.999
lr = 1e-3
eps = 1e-8
steps = 10000
report_steps = [1000, 5000, 10000]

rng = torch.Generator().manual_seed(303)
grads = [torch.randn(n, generator=rng) * 0.01 for _ in range(steps)]

configs = {
    "fp32":            (None, None),
    "log_adapt+fp32M": (lambda v: qd_log_adaptive(v, 2048), None),
    "bnb+fp32M":       (lambda v: qd_bnb(v, 256),           None),
    "log_adapt+UF8":   (lambda v: qd_log_adaptive(v, 2048), lambda m: qd_uf(m, 256, 8)),
    "log_adapt+D8":    (lambda v: qd_log_adaptive(v, 2048), lambda m: qd_dynamic_signed(m, 256, 8)),
    "bnb+D8":          (lambda v: qd_bnb(v, 256),           lambda m: qd_dynamic_signed(m, 256, 8)),
}

params = {name: torch.zeros(n) for name in configs}
ms = {name: torch.zeros(n) for name in configs}
vs = {name: torch.zeros(n) for name in configs}

print(f"C5: Parameter drift ({steps} steps, full Adam)")
print(f"\n  {'step':<8}", end="")
for name in configs:
    if name != "fp32":
        print(f" {name:>18}", end="")
print()
print("  " + "-" * (8 + 19 * (len(configs) - 1)))

for t in range(steps):
    g = grads[t]
    step = t + 1
    bc1 = 1.0 - beta1 ** step
    bc2 = 1.0 - beta2 ** step
    step_size = lr * math.sqrt(bc2) / bc1
    eps_corr = eps * math.sqrt(bc2)

    for name, (v_qfn, m_qfn) in configs.items():
        vs[name] = beta2 * vs[name] + (1 - beta2) * g.square()
        ms[name] = beta1 * ms[name] + (1 - beta1) * g
        inv_std = 1.0 / (vs[name].sqrt() + eps_corr)
        params[name] -= step_size * ms[name] * inv_std
        if v_qfn is not None:
            vs[name] = v_qfn(vs[name])
        if m_qfn is not None:
            ms[name] = m_qfn(ms[name])

    if step in report_steps:
        p_ref = params["fp32"]
        p_norm = p_ref.norm().item()
        print(f"  {step:<8}", end="")
        for name in configs:
            if name != "fp32":
                drift = ((params[name] - p_ref).norm() / max(p_norm, 1e-30)).item()
                print(f" {drift*100:>17.4f}%", end="")
        print()

C5: Parameter drift (10000 steps, full Adam)

  step        log_adapt+fp32M          bnb+fp32M      log_adapt+UF8       log_adapt+D8             bnb+D8
  -------------------------------------------------------------------------------------------------------
  1000                0.3626%            1.3331%            1.5596%            2.3122%            2.6480%
  5000                0.3800%            2.6821%            1.6808%            2.3515%            3.5779%
  10000               0.3909%            3.2756%            1.8189%            2.4305%            4.0851%


## §D Engineering Selection

Block size sensitivity, outlier robustness, memory-precision tradeoff.

In [18]:
# D1: Block size sensitivity (single-step update error)
# Uses Adafactor-style metric (rsqrt, no eps damping) to stress-test
# block size sensitivity under the stricter inv_std path.

dists = ["narrow_0.5dec", "medium_10dec", "wide_20dec", "outlier"]
block_sizes = [128, 256, 512, 1024, 2048, 4096]
n = 8192

methods = {
    "log_adapt": qd_log_adaptive,
    "bnb":       qd_bnb,
}

print("D1: Block size sensitivity (single-step update error %)")

for dname in dists:
    v = gen_v(dname, n, seed=42)
    rng = torch.Generator().manual_seed(7)
    grad = torch.randn(n, generator=rng) * 0.01
    print(f"\n  {dname}")
    print(f"  {'method':<12}", end="")
    for bs in block_sizes:
        print(f" bs={bs:<5}", end="")
    print()
    print("  " + "-" * (12 + 8 * len(block_sizes)))
    for mname, mfn in methods.items():
        print(f"  {mname:<12}", end="")
        for bs in block_sizes:
            v_q = mfn(v, bs)
            ue = update_error_adafactor(grad, v, v_q) * 100
            print(f" {fmt_err(ue)}", end="")
        print()

D1: Block size sensitivity (single-step update error %)

  narrow_0.5dec
  method       bs=128   bs=256   bs=512   bs=1024  bs=2048  bs=4096 
  ------------------------------------------------------------
  log_adapt        0.031     0.033     0.035     0.037     0.039     0.041
  bnb              0.134     0.136     0.139     0.142     0.143     0.146

  medium_10dec
  method       bs=128   bs=256   bs=512   bs=1024  bs=2048  bs=4096 
  ------------------------------------------------------------
  log_adapt        0.477     0.530     0.598     0.669     0.688     0.728
  bnb              5.558     6.428     7.469     8.021     9.249    10.113

  wide_20dec
  method       bs=128   bs=256   bs=512   bs=1024  bs=2048  bs=4096 
  ------------------------------------------------------------
  log_adapt        0.323     0.381     0.524     0.618     0.704     0.804
  bnb              >100%     >100%     >100%     >100%     >100%     >100%

  outlier
  method       bs=128   bs=256   bs=512 

In [19]:
# D2: Outlier sensitivity (single roundtrip)
# Inject outliers into a base V distribution, measure degradation by block size.

n = 8192
n_trials = 20
block_sizes = [256, 512, 1024, 2048]
outlier_fracs = [0.0, 0.01, 0.05]

methods = {
    "log_adapt": qd_log_adaptive,
    "bnb":       qd_bnb,
}

print("D2: Outlier sensitivity (mean abs relative error %)")
print(f"\n  {'method':<12} {'bs':<6}", end="")
for of in outlier_fracs:
    print(f" {'out='+str(of):>10}", end="")
print()
print("  " + "-" * (12 + 6 + 11 * len(outlier_fracs)))

for mname, mfn in methods.items():
    for bs in block_sizes:
        print(f"  {mname:<12} {bs:<6}", end="")
        for outlier_frac in outlier_fracs:
            errs = []
            for trial in range(n_trials):
                rng = torch.Generator().manual_seed(trial * 100 + bs)
                log_v = torch.randn(n, generator=rng) * 1.0 - 13.3
                n_out = int(n * outlier_frac)
                if n_out > 0:
                    log_v[:n_out] = torch.randn(n_out, generator=rng) * 0.5 - 3.0
                v = torch.pow(2.0, log_v)
                v_deq = mfn(v, bs)
                errs.append(((v_deq / v) - 1.0).abs().mean().item())
            mean_err = sum(errs) / len(errs) * 100
            print(f" {mean_err:>9.4f}%", end="")
        print()

D2: Outlier sensitivity (mean abs relative error %)

  method       bs        out=0.0   out=0.01   out=0.05
  ---------------------------------------------------
  log_adapt    256       0.3806%    0.3994%    0.3927%
  log_adapt    512       0.4112%    0.4465%    0.4462%
  log_adapt    1024      0.4367%    0.5087%    0.5095%
  log_adapt    2048      0.4711%    0.6086%    0.6110%
  bnb          256       0.8308%    0.9094%    0.8622%
  bnb          512       0.8521%    1.0489%    0.8911%
  bnb          1024      0.8645%    1.2984%    1.1697%
  bnb          2048      0.8885%    1.7758%    1.7273%


In [20]:
# D3: V memory estimation (1.1B parameters)
# log_adapt stores 2 floats per block (scale + min_log); bnb stores 1 (absmax).

n_params = 1_100_000_000

configs = [
    ("fp32",              32.0),
    ("bnb bs=256",        8 + 32/256),
    ("log_adapt bs=256",  8 + 64/256),
    ("log_adapt bs=1024", 8 + 64/1024),
    ("log_adapt bs=2048", 8 + 64/2048),
]

print(f"D3: V memory estimation ({n_params/1e9:.1f}B parameters)")
print(f"  log_adapt: 8-bit data + 2 floats/block (scale, min_log)")
print(f"  bnb:       8-bit data + 1 float/block (absmax)")
print(f"\n  {'config':<22} {'bits/elem':>10} {'V MB':>10} {'vs bnb_256':>12}")
print("  " + "-" * 58)

bnb_bits = configs[1][1]
bnb_mb = n_params * bnb_bits / 8 / 1024 / 1024
for name, bits in configs:
    mb = n_params * bits / 8 / 1024 / 1024
    if name.startswith("bnb"):
        delta = "baseline"
    else:
        saved = mb - bnb_mb
        delta = f"{saved:+.1f} MB"
    print(f"  {name:<22} {bits:>9.3f} {mb:>9.1f} {delta:>12}")

print(f"\n  M memory (independent of V): UF4 bs=128: 4.250, "
      f"UF8/D8 bs=256: 8.125 bits/elem")

D3: V memory estimation (1.1B parameters)
  log_adapt: 8-bit data + 2 floats/block (scale, min_log)
  bnb:       8-bit data + 1 float/block (absmax)

  config                  bits/elem       V MB   vs bnb_256
  ----------------------------------------------------------
  fp32                      32.000    4196.2   +3130.7 MB
  bnb bs=256                 8.125    1065.4     baseline
  log_adapt bs=256           8.250    1081.8     +16.4 MB
  log_adapt bs=1024          8.062    1057.2      -8.2 MB
  log_adapt bs=2048          8.031    1053.1     -12.3 MB

  M memory (independent of V): UF4 bs=128: 4.250, UF8/D8 bs=256: 8.125 bits/elem


In [21]:
# D4: Precision-memory Pareto front (5000-step EMA, V drift)
# V-only quantization (M is fp32). Metric: relative L2 drift of V.

n = 8192
beta_val = 0.001
steps = 5000

rng = torch.Generator().manual_seed(456)
grads = [torch.randn(n, generator=rng) * 0.01 for _ in range(steps)]

v_ref = torch.zeros(n)
for g in grads:
    v_ref = (1 - beta_val) * v_ref + beta_val * g.square()

configs = []
for bs in [128, 256, 512, 1024, 2048, 4096]:
    configs.append((f"bnb_{bs}",       bs, lambda v, b=bs: qd_bnb(v, b),          8 + 32/bs))
    configs.append((f"log_adapt_{bs}", bs, lambda v, b=bs: qd_log_adaptive(v, b),  8 + 64/bs))

# Compute all results first
results = []
for name, bs, qfn, v_bits in configs:
    v_stored = torch.zeros(n)
    for g in grads:
        v_new = (1 - beta_val) * v_stored + beta_val * g.square()
        v_stored = qfn(v_new)
    drift = rel_l2(v_ref, v_stored) * 100
    results.append((name, v_bits, drift))

bnb_256_drift = next(d for nm, _, d in results if nm == "bnb_256")
bnb_256_bits = next(b for nm, b, _ in results if nm == "bnb_256")

print(f"D4: Pareto front ({steps}-step EMA, V-only, M=fp32)")
print(f"\n  {'config':<18} {'V bits':>8} {'drift%':>10} {'vs bnb_256':>12} {'dominates?':>12}")
print("  " + "-" * 64)

for name, v_bits, drift in results:
    ratio = drift / bnb_256_drift
    better_prec = drift < bnb_256_drift
    less_mem = v_bits < bnb_256_bits
    if name == "bnb_256":
        dom = "(reference)"
    elif better_prec and less_mem:
        dom = "both"
    elif better_prec:
        dom = "precision"
    elif less_mem:
        dom = "memory"
    else:
        dom = "no"
    print(f"  {name:<18} {v_bits:>7.3f} {drift:>9.3f} {ratio:>11.2f}x {dom:>12}")

D4: Pareto front (5000-step EMA, V-only, M=fp32)

  config               V bits     drift%   vs bnb_256   dominates?
  ----------------------------------------------------------------
  bnb_128              8.250     6.449        0.78x    precision
  log_adapt_128        8.500     0.510        0.06x    precision
  bnb_256              8.125     8.243        1.00x  (reference)
  log_adapt_256        8.250     0.577        0.07x    precision
  bnb_512              8.062     8.598        1.04x       memory
  log_adapt_512        8.125     0.643        0.08x    precision
  bnb_1024             8.031     8.847        1.07x       memory
  log_adapt_1024       8.062     0.710        0.09x         both
  bnb_2048             8.016    12.259        1.49x       memory
  log_adapt_2048       8.031     0.814        0.10x         both
  bnb_4096             8.008    16.508        2.00x       memory
  log_adapt_4096       8.016     0.838        0.10x         both


## §E Design Decision Ablation

Each experiment isolates one design choice. All quantization functions from §0.
Metric path noted per experiment (adam / adafactor / no_clamp).

In [22]:
# E1: V zero reservation — 1+255 vs 0+255
# Metric: adafactor path and no_clamp path

dists = ["narrow_0.5dec", "medium_10dec", "wide_20dec", "ema_realistic", "sparse_90pct"]
methods = {
    "1+255 (kernel)": qd_log_adaptive,
    "0+255 (no zero)": qd_log_adaptive_nz,
}
bs = 2048

def fmt_err(val):
    return f"{val:>9.3f}" if val <= 100 else "    >100%"

print(f"E1: V zero reservation (block_size={bs})")
print(f"  {'distribution':<26} {'method':<18} {'rel_L2%':>9} {'upd_adaf%':>11} {'upd_noclip%':>13}")
print("  " + "-" * 81)

for dname in dists:
    v = gen_v(dname, 8192, seed=42)
    rng = torch.Generator().manual_seed(7)
    grad = torch.randn(8192, generator=rng) * 0.01
    n_zero = (v == 0).sum().item()
    label = f"{dname} (z={n_zero})"
    for mname, mfn in methods.items():
        v_q = mfn(v, bs)
        l2 = rel_l2(v, v_q) * 100
        ue_ada = update_error_adafactor(grad, v, v_q) * 100
        ue_nc = update_error_no_clamp(grad, v, v_q) * 100
        print(f"  {label:<26} {mname:<18} {l2:>8.3f} {fmt_err(ue_ada)} {fmt_err(ue_nc)}")
    print()

E1: V zero reservation (block_size=2048)
  distribution               method               rel_L2%   upd_adaf%   upd_noclip%
  ---------------------------------------------------------------------------------
  narrow_0.5dec (z=0)        1+255 (kernel)        0.079     0.039     0.039
  narrow_0.5dec (z=0)        0+255 (no zero)       0.078     0.039     0.039

  medium_10dec (z=0)         1+255 (kernel)        1.373     0.688     0.688
  medium_10dec (z=0)         0+255 (no zero)       1.367     0.701     0.701

  wide_20dec (z=0)           1+255 (kernel)        1.526     0.704     0.704
  wide_20dec (z=0)           0+255 (no zero)       1.506     0.716     0.716

  ema_realistic (z=0)        1+255 (kernel)        0.049     0.025     0.025
  ema_realistic (z=0)        0+255 (no zero)       0.049     0.024     0.024

  sparse_90pct (z=7373)      1+255 (kernel)        0.243     0.000     0.122
  sparse_90pct (z=7373)      0+255 (no zero)       8.963     0.000     >100%



In [23]:
# E2: V fixed vs adaptive min_log (compressed)
# Metric: adafactor path

dists = ["narrow_0.5dec", "medium_10dec", "wide_20dec", "sparse_90pct"]
fixed_logs = [-126, -53, -40]
bs = 2048

print(f"E2: V fixed vs adaptive min_log (block_size={bs})")
print(f"  {'distribution':<18}", end="")
for ml in fixed_logs:
    print(f" {'fix'+str(ml):>10}", end="")
print(f" {'adaptive':>10}")
print("  " + "-" * (18 + 11 * (len(fixed_logs) + 1)))

for dname in dists:
    v = gen_v(dname, 8192, seed=42)
    grad = torch.randn(8192, generator=torch.Generator().manual_seed(7)) * 0.01
    print(f"  {dname:<18}", end="")
    for ml in fixed_logs:
        v_q = qd_log_fixed(v, bs, float(ml))
        ue = update_error_adafactor(grad, v, v_q) * 100
        print(f" {ue:>9.3f}%", end="")
    v_q = qd_log_adaptive(v, bs)
    ue = update_error_adafactor(grad, v, v_q) * 100
    print(f" {ue:>9.3f}%")

E2: V fixed vs adaptive min_log (block_size=2048)
  distribution          fix-126     fix-53     fix-40   adaptive
  --------------------------------------------------------------
  narrow_0.5dec          4.200%     1.568%     1.052%     0.039%
  medium_10dec           5.140%     1.835%     1.574%     0.688%
  wide_20dec             2.934%     1.993%    44.505%     0.704%
  sparse_90pct           0.000%   100.000%   100.000%     0.000%


In [24]:
# E3: V 4-bit vs 8-bit
# Single-step + 5000-step EMA drift. Metric: adafactor path.

n = 8192
bs = 2048

methods_4bit = {
    "log_adapt_4 (15+z)": lambda v: qd_log_adaptive_generic(v, bs, 16, zero_reserved=True),
    "dynmap_4 (16 lvl)":  lambda v: qd_dynmap(v, bs, 4),
    "uniform_4 (16 lvl)": lambda v: qd_uniform(v, bs, 16),
}
methods_8bit = {
    "log_adapt_8 (255+z)": lambda v: qd_log_adaptive(v, bs),
    "bnb_8 (256 lvl)":     lambda v: qd_bnb(v, bs),
}

def fmt_err(val):
    return f"{val:>9.3f}" if val <= 100 else "    >100%"

# --- Single-step ---
print(f"E3a: Single-step V quantization (block_size={bs})")
print(f"  {'distribution':<18} {'method':<22} {'rel_L2%':>9} {'upd_err%':>10}")
print("  " + "-" * 63)

for dname in ["narrow_0.5dec", "medium_10dec", "wide_20dec"]:
    v = gen_v(dname, n, seed=42)
    rng = torch.Generator().manual_seed(7)
    grad = torch.randn(n, generator=rng) * 0.01
    for mname, mfn in {**methods_4bit, **methods_8bit}.items():
        v_q = mfn(v)
        l2 = rel_l2(v, v_q) * 100
        ue = update_error_adafactor(grad, v, v_q) * 100
        print(f"  {dname:<18} {mname:<22} {l2:>8.3f} {fmt_err(ue)}")
    print()

# --- EMA drift (5000 steps, correct feedback loop) ---
beta_val = 0.001
steps = 5000
rng = torch.Generator().manual_seed(123)
grads = [torch.randn(n, generator=rng) * 0.01 for _ in range(steps)]

v_ref = torch.zeros(n)
for g in grads:
    v_ref = (1 - beta_val) * v_ref + beta_val * g.square()

print(f"E3b: EMA drift after {steps} steps (beta={beta_val})")
print(f"  {'method':<22} {'drift%':>10} {'bits/elem':>10}")
print("  " + "-" * 46)

all_methods = {**methods_4bit, **methods_8bit}
bits_map = {
    "log_adapt_4 (15+z)": 4 + 64/bs,
    "dynmap_4 (16 lvl)":  4 + 32/bs,
    "uniform_4 (16 lvl)": 4 + 32/bs,
    "log_adapt_8 (255+z)": 8 + 64/bs,
    "bnb_8 (256 lvl)":     8 + 32/bs,
}

for mname, mfn in all_methods.items():
    v_stored = torch.zeros(n)
    for g in grads:
        v_new = (1 - beta_val) * v_stored + beta_val * g.square()
        v_stored = mfn(v_new)
    drift = rel_l2(v_ref, v_stored) * 100
    bits = bits_map[mname]
    print(f"  {mname:<22} {drift:>9.3f} {bits:>9.3f}")

E3a: Single-step V quantization (block_size=2048)
  distribution       method                   rel_L2%   upd_err%
  ---------------------------------------------------------------
  narrow_0.5dec      log_adapt_4 (15+z)        1.438     0.724
  narrow_0.5dec      dynmap_4 (16 lvl)         4.487     2.330
  narrow_0.5dec      uniform_4 (16 lvl)        2.639     1.363
  narrow_0.5dec      log_adapt_8 (255+z)       0.079     0.039
  narrow_0.5dec      bnb_8 (256 lvl)           0.279     0.143

  medium_10dec       log_adapt_4 (15+z)       22.652    13.266
  medium_10dec       dynmap_4 (16 lvl)         9.787     >100%
  medium_10dec       uniform_4 (16 lvl)       19.144     >100%
  medium_10dec       log_adapt_8 (255+z)       1.373     0.688
  medium_10dec       bnb_8 (256 lvl)           0.657     9.249

  wide_20dec         log_adapt_4 (15+z)       22.420    13.832
  wide_20dec         dynmap_4 (16 lvl)         4.729     >100%
  wide_20dec         uniform_4 (16 lvl)        8.860     >100

In [25]:
# E4: M uniform vs dynamic map (8-bit, correct codebook)

n = 8192
block_sizes = [128, 256]

rng_sparse = torch.Generator().manual_seed(99)
m_configs = [
    ("M@step100",  lambda: gen_m_ema(n, 100, 0.9, 0.01, seed=42)),
    ("M@step1000", lambda: gen_m_ema(n, 1000, 0.9, 0.01, seed=42)),
    ("M sparse",   lambda: gen_m_ema(n, 500, 0.9, 0.01, seed=42)
                           * (torch.rand(n, generator=rng_sparse) < 0.1).float()),
]

methods = {
    "UF8": lambda v, bs: qd_uf(v, bs, 8),
    "D8":  lambda v, bs: qd_dynamic_signed(v, bs, 8),
}

print("E4: M 8-bit — uniform vs dynamic map")
for cname, cfn in m_configs:
    m = cfn()
    print(f"\n  {cname} (|m|_max={m.abs().max():.2e})")
    print(f"  {'bs':<6} {'method':<10} {'rel_L2%':>9} {'cos_sim':>10} {'signflip%':>11} {'asym_bias%':>12}")
    print("  " + "-" * 62)
    for bs in block_sizes:
        for mname, mfn in methods.items():
            m_q = mfn(m, bs)
            l2 = rel_l2_signed(m, m_q) * 100
            cs = cosine_sim(m, m_q)
            sf = sign_flip_rate(m, m_q) * 100
            ab = asymmetry_bias(m, m_q) * 100
            print(f"  {bs:<6} {mname:<10} {l2:>8.4f} {cs:>9.6f} {sf:>10.4f} {ab:>+11.4f}")

E4: M 8-bit — uniform vs dynamic map

  M@step100 (|m|_max=9.78e-03)
  bs     method       rel_L2%    cos_sim   signflip%   asym_bias%
  --------------------------------------------------------------
  128    UF8          0.6466  0.999979     0.0000     -0.2171
  128    D8           1.0054  0.999949     0.0000     +0.0052
  256    UF8          0.6954  0.999976     0.0000     -0.2803
  256    D8           1.0711  0.999943     0.0000     +0.0047

  M@step1000 (|m|_max=1.02e-02)
  bs     method       rel_L2%    cos_sim   signflip%   asym_bias%
  --------------------------------------------------------------
  128    UF8          0.6733  0.999977     0.0000     -0.0975
  128    D8           1.0525  0.999945     0.0000     -0.0427
  256    UF8          0.7126  0.999975     0.0000     -0.0797
  256    D8           1.1134  0.999938     0.0000     -0.0378

  M sparse (|m|_max=8.24e-03)
  bs     method       rel_L2%    cos_sim   signflip%   asym_bias%
  -----------------------------------------

In [26]:
# E5: M 4-bit — uniform vs dynamic map (correct codebook)
# Note: asymmetry_bias is sensitive to near-zero values (denominator ~0), for reference only.

n = 8192
block_sizes = [128, 256]

m_configs = [
    ("M@step100",  lambda: gen_m_ema(n, 100, 0.9, 0.01, seed=42)),
    ("M@step1000", lambda: gen_m_ema(n, 1000, 0.9, 0.01, seed=42)),
]

methods = {
    "UF4": lambda v, bs: qd_uf(v, bs, 4),
    "D4":  lambda v, bs: qd_dynamic_signed(v, bs, 4),
}

print("E5: M 4-bit — uniform vs dynamic map")
for cname, cfn in m_configs:
    m = cfn()
    print(f"\n  {cname} (|m|_max={m.abs().max():.2e})")
    print(f"  {'bs':<6} {'method':<10} {'rel_L2%':>9} {'cos_sim':>10} {'signflip%':>11} {'asym_bias%':>12}")
    print("  " + "-" * 62)
    for bs in block_sizes:
        for mname, mfn in methods.items():
            m_q = mfn(m, bs)
            l2 = rel_l2_signed(m, m_q) * 100
            cs = cosine_sim(m, m_q)
            sf = sign_flip_rate(m, m_q) * 100
            ab = asymmetry_bias(m, m_q) * 100
            print(f"  {bs:<6} {mname:<10} {l2:>8.4f} {cs:>9.6f} {sf:>10.4f} {ab:>+11.4f}")

# EMA drift (5000 steps)
print(f"\n  EMA drift (5000 steps, beta1=0.9):")
steps = 5000
rng = torch.Generator().manual_seed(123)
grads = [torch.randn(n, generator=rng) * 0.01 for _ in range(steps)]

m_ref = torch.zeros(n)
for g in grads:
    m_ref = 0.9 * m_ref + 0.1 * g

print(f"  {'method':<12} {'drift%':>10}")
print("  " + "-" * 24)
for mname, mfn in [("UF4_128", lambda v: qd_uf(v, 128, 4)),
                    ("D4_128",  lambda v: qd_dynamic_signed(v, 128, 4)),
                    ("UF8_128", lambda v: qd_uf(v, 128, 8)),
                    ("D8_128",  lambda v: qd_dynamic_signed(v, 128, 8))]:
    m_q = torch.zeros(n)
    for g in grads:
        m_new = 0.9 * m_q + 0.1 * g
        m_q = mfn(m_new)
    drift = rel_l2_signed(m_ref, m_q) * 100
    print(f"  {mname:<12} {drift:>9.4f}")

E5: M 4-bit — uniform vs dynamic map

  M@step100 (|m|_max=9.78e-03)
  bs     method       rel_L2%    cos_sim   signflip%   asym_bias%
  --------------------------------------------------------------
  128    UF4         10.4052  0.994589     0.0000     -0.0631
  128    D4          14.5992  0.989388     0.0000   -144.7536
  256    UF4         11.1645  0.993803     0.0000     -0.3431
  256    D4          15.3917  0.988190     0.0000   -174.1673

  M@step1000 (|m|_max=1.02e-02)
  bs     method       rel_L2%    cos_sim   signflip%   asym_bias%
  --------------------------------------------------------------
  128    UF4         10.8035  0.994159     0.0000     +0.1482
  128    D4          15.0776  0.988702     0.0000     -1.8707
  256    UF4         11.4984  0.993407     0.0000     +0.4983
  256    D4          15.9915  0.987267     0.0000     -2.1389

  EMA drift (5000 steps, beta1=0.9):
  method           drift%
  ------------------------
  UF4_128        23.8333
  D4_128         34.2519

In [27]:
# E6: V quantization scheme comparison
# NF4/FP4 are weight-designed, used as controls.

n = 8192
bs = 2048

methods = {
    "log_adapt_8 (255+z)":  (lambda v: qd_log_adaptive(v, bs),        "8-bit, 255+zero"),
    "log_adapt_4 (15+z)":   (lambda v: qd_log_adaptive_generic(v, bs, 16, zero_reserved=True), "4-bit, 15+zero"),
    "NF4 (9 non-neg)":      (lambda v: qd_nf4_v(v, bs),               "4-bit, 9 non-neg"),
    "FP4 (8 mag)":          (lambda v: qd_fp4_v(v, bs),               "4-bit, 8 magnitude"),
    "dynmap_4 (16 lvl)":    (lambda v: qd_dynmap(v, bs, 4),           "4-bit, 16 unsigned"),
    "uniform_4 (16 lvl)":   (lambda v: qd_uniform(v, bs, 16),         "4-bit, 16 linear"),
    "uniform_8 (256 lvl)":  (lambda v: qd_uniform(v, bs, 256),        "8-bit, 256 linear"),
}

dists = ["narrow_0.5dec", "medium_10dec", "wide_20dec", "ema_realistic", "sparse_90pct"]

def fmt_err(val):
    return f"{val:>9.3f}" if val <= 100 else "    >100%"

print(f"E6: V quantization scheme comparison (block_size={bs})")
print(f"  {'distribution':<18} {'method':<24} {'levels':<20} {'rel_L2%':>9} {'upd_err%':>10}")
print("  " + "-" * 85)

for dname in dists:
    v = gen_v(dname, n, seed=42)
    rng = torch.Generator().manual_seed(7)
    grad = torch.randn(n, generator=rng) * 0.01
    for mname, (mfn, levels_desc) in methods.items():
        v_q = mfn(v)
        l2 = rel_l2(v, v_q) * 100
        ue = update_error_adafactor(grad, v, v_q) * 100
        print(f"  {dname:<18} {mname:<24} {levels_desc:<20} {l2:>8.3f} {fmt_err(ue)}")
    print()

E6: V quantization scheme comparison (block_size=2048)
  distribution       method                   levels                 rel_L2%   upd_err%
  -------------------------------------------------------------------------------------
  narrow_0.5dec      log_adapt_8 (255+z)      8-bit, 255+zero         0.079     0.039
  narrow_0.5dec      log_adapt_4 (15+z)       4-bit, 15+zero          1.438     0.724
  narrow_0.5dec      NF4 (9 non-neg)          4-bit, 9 non-neg        8.017     3.911
  narrow_0.5dec      FP4 (8 mag)              4-bit, 8 magnitude     11.094     5.409
  narrow_0.5dec      dynmap_4 (16 lvl)        4-bit, 16 unsigned      4.487     2.330
  narrow_0.5dec      uniform_4 (16 lvl)       4-bit, 16 linear        2.639     1.363
  narrow_0.5dec      uniform_8 (256 lvl)      8-bit, 256 linear       0.155     0.081

  medium_10dec       log_adapt_8 (255+z)      8-bit, 255+zero         1.373     0.688
  medium_10dec       log_adapt_4 (15+z)       4-bit, 15+zero         22.652    1

## §F Optimizer Path Analysis
 
 Motivated by training observations: the Adafactor path (factored V, no M,
 RMS scaling) shows a larger eval gap than the Adam path under identical
 quantization. This section isolates the contributing factors.
 
 State trace reference (step 10000, Adafactor factored V, TinyLlama-1.1B):
   - emb row V:     BR med = 81.5, log2 p5 = -95.1, p95 = -21.5
   - lm_head row V: BR med = 34.9, log2 p5 = -50.9, p95 = -21.4
   - attn/mlp row V: BR med = 0.7 - 9.1
 
 F1: RMS clipping (d parameter) stabilizes quantized factored V. 
 F2: Factored V structure vs full-rank V under heterogeneous rows.  
 F3: Extreme-BR layers (emb, lm_head) — log_adapt vs bnb.  
 F4: Secondary factors (RMS feedback, dynamic beta, gradient noise).  
 F5: CAME residual under extreme BR (V/M quantization, BC effect).  
 F6: 8-bit vs 16-bit adaptive log V (upper-bound exploration). 

In [28]:
# F1: RMS clipping effect on quantized factored V
# 2D factored Adafactor with gradient bursts (periodic distribution shifts).
# Periodic bursts simulate abrupt distribution shifts; in practice, shifts
# are gradual but the stabilizing mechanism is the same.
# Compares d=0 (no clipping) vs d=1.0 (clipping) under fp32 and quantized V.

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
R, C = 2048, 2048
steps = 5000
eps1 = 1e-30
eps2 = 1e-3
lr = 1e-3
burst_interval = 200

qfn_v = lambda v: qd_log_adaptive(v, 2048)

def run_factored_adafactor(d_val, use_quant):
    rng = torch.Generator(device=device).manual_seed(42)
    p = torch.randn(R, C, generator=rng, device=device) * 0.02
    row = torch.zeros(R, device=device)
    col = torch.zeros(C, device=device)
    for t in range(steps):
        step = t + 1
        beta2t = 1.0 - step ** (-0.8)
        g = torch.randn(R, C, generator=rng, device=device) * 0.01
        if t % burst_interval == 0:
            g = g + torch.randn(R, C, generator=rng, device=device) * 0.05
        g_sq_eps = g.square() + eps1
        row_new = beta2t * row + (1 - beta2t) * g_sq_eps.mean(dim=-1)
        col_new = beta2t * col + (1 - beta2t) * g_sq_eps.mean(dim=0)
        if use_quant:
            row = qfn_v(row_new)
            col = qfn_v(col_new)
        else:
            row = row_new
            col = col_new
        row_mean = row.mean().clamp(min=eps1)
        update = g * (row / row_mean).rsqrt().unsqueeze(-1) * col.rsqrt().unsqueeze(0)
        rms_u = (update.norm() / math.sqrt(R * C)).item()
        if d_val > 0:
            update = update / max(rms_u / d_val, 1.0)
        rms_p = max(eps2, (p.norm() / math.sqrt(R * C)).item())
        p = p - rms_p * lr * update
    return p

print(f"F1: RMS clipping effect ({steps} steps, R={R}, C={C}, factored V, device={device})")
print(f"  Gradient bursts every {burst_interval} steps")

p_ref = run_factored_adafactor(d_val=1.0, use_quant=False)
p_fp32_noclip = run_factored_adafactor(d_val=0.0, use_quant=False)
p_q_noclip = run_factored_adafactor(d_val=0.0, use_quant=True)
p_q_clip = run_factored_adafactor(d_val=1.0, use_quant=True)

ref_norm = p_ref.norm().item()
for name, p in [("fp32, d=1 (clip)", p_ref),
                ("fp32, d=0 (no clip)", p_fp32_noclip),
                ("quant, d=0 (no clip)", p_q_noclip),
                ("quant, d=1 (clip)", p_q_clip)]:
    drift = ((p - p_ref).norm() / ref_norm).item() * 100
    print(f"  {name:<24} drift: {drift:.4f}%")

F1: RMS clipping effect (5000 steps, R=2048, C=2048, factored V, device=cuda)
  Gradient bursts every 200 steps
  fp32, d=1 (clip)         drift: 0.0000%
  fp32, d=0 (no clip)      drift: 1.8219%
  quant, d=0 (no clip)     drift: 1.8219%
  quant, d=1 (clip)        drift: 0.0011%


In [3]:
# F2: Factored V vs full-rank V under heterogeneous rows
# Tests whether the factored approximation amplifies quantization errors.
# Compares log_adapt and bnb within each V structure.
# Note: B2b uses homogeneous rows (BR < 1); F2 uses heterogeneous rows
# (BR = 36.7) matching extreme-BR layers (emb, lm_head).

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
R, C = 2048, 2048
steps = 5000
eps1 = 1e-30
beta2 = 0.999
bv = 1.0 - beta2

rng = torch.Generator(device=device).manual_seed(42)
row_log_scale = torch.zeros(R, device=device)
n_hot = int(R * 0.2)
row_log_scale[:n_hot] = torch.randn(n_hot, generator=rng, device=device) * 1.0 - 10.0
row_log_scale[n_hot:] = torch.randn(R - n_hot, generator=rng, device=device) * 1.0 - 40.0
row_scale = torch.pow(2.0, row_log_scale)
br = (torch.log2(row_scale.max()) - torch.log2(row_scale.min())).item()

print(f"F2: Factored vs full-rank V ({steps} steps, R={R}, C={C}, device={device})")
print(f"  Row heterogeneity: {n_hot} hot / {R - n_hot} cold, BR = {br:.1f} decades")

# fp32 reference
rng_ref = torch.Generator(device=device).manual_seed(42)
row_ref = torch.zeros(R, device=device); col_ref = torch.zeros(C, device=device)
for t in range(steps):
    g = torch.randn(R, C, generator=rng_ref, device=device) * 0.01 * row_scale.unsqueeze(1)
    g_sq = g.square()
    row_ref = beta2 * row_ref + bv * (g_sq.mean(dim=-1) + eps1)
    col_ref = beta2 * col_ref + bv * (g_sq.mean(dim=0) + eps1)
V_ref = row_ref.unsqueeze(1) * col_ref.unsqueeze(0) / row_ref.mean().clamp(min=eps1)

# 2D factored (log_adapt)
rng_f1 = torch.Generator(device=device).manual_seed(42)
row_q1 = torch.zeros(R, device=device); col_q1 = torch.zeros(C, device=device)
for t in range(steps):
    g = torch.randn(R, C, generator=rng_f1, device=device) * 0.01 * row_scale.unsqueeze(1)
    g_sq = g.square()
    row_q1 = qd_log_adaptive(beta2 * row_q1 + bv * (g_sq.mean(dim=-1) + eps1), 2048)
    col_q1 = qd_log_adaptive(beta2 * col_q1 + bv * (g_sq.mean(dim=0) + eps1), 2048)
V_2d_la = row_q1.unsqueeze(1) * col_q1.unsqueeze(0) / row_q1.mean().clamp(min=eps1)

# 2D factored (bnb)
rng_f2 = torch.Generator(device=device).manual_seed(42)
row_q2 = torch.zeros(R, device=device); col_q2 = torch.zeros(C, device=device)
for t in range(steps):
    g = torch.randn(R, C, generator=rng_f2, device=device) * 0.01 * row_scale.unsqueeze(1)
    g_sq = g.square()
    row_q2 = qd_bnb(beta2 * row_q2 + bv * (g_sq.mean(dim=-1) + eps1), 256)
    col_q2 = qd_bnb(beta2 * col_q2 + bv * (g_sq.mean(dim=0) + eps1), 256)
V_2d_bnb = row_q2.unsqueeze(1) * col_q2.unsqueeze(0) / row_q2.mean().clamp(min=eps1)

# 1D full-rank (log_adapt)
rng_1 = torch.Generator(device=device).manual_seed(42)
v_1d = torch.zeros(R, C, device=device)
for t in range(steps):
    g = torch.randn(R, C, generator=rng_1, device=device) * 0.01 * row_scale.unsqueeze(1)
    g_sq = g.square()
    v_1d = qd_log_adaptive(beta2 * v_1d + bv * (g_sq + eps1), 2048)

# 1D full-rank (bnb)
rng_1b = torch.Generator(device=device).manual_seed(42)
v_1d_bnb = torch.zeros(R, C, device=device)
for t in range(steps):
    g = torch.randn(R, C, generator=rng_1b, device=device) * 0.01 * row_scale.unsqueeze(1)
    g_sq = g.square()
    v_1d_bnb = qd_bnb(beta2 * v_1d_bnb + bv * (g_sq + eps1), 256)

# V drift (L2 metric)
print(f"\n  {'config':<32} {'V_drift%':>10}")
print("  " + "-" * 44)
for name, vq in [("2D factored (log_adapt)", V_2d_la),
                 ("2D factored (bnb)", V_2d_bnb),
                 ("1D full-rank (log_adapt)", v_1d),
                 ("1D full-rank (bnb)", v_1d_bnb)]:
    drift = rel_l2(V_ref.flatten(), vq.flatten()) * 100
    print(f"  {name:<32} {drift:>9.4f}")

# Single-step update error (rsqrt metric)
g_test = torch.randn(R, C, generator=torch.Generator(device=device).manual_seed(99),
                     device=device) * 0.01 * row_scale.unsqueeze(1)
u_ref = g_test * torch.rsqrt(V_ref.clamp(min=MIN_VAL))
print(f"\n  {'config':<32} {'upd_err%':>10}")
print("  " + "-" * 44)
for name, vq in [("2D factored (log_adapt)", V_2d_la),
                 ("2D factored (bnb)", V_2d_bnb),
                 ("1D full-rank (log_adapt)", v_1d),
                 ("1D full-rank (bnb)", v_1d_bnb)]:
    u_q = g_test * torch.rsqrt(vq.clamp(min=MIN_VAL))
    err = ((u_q - u_ref).norm() / u_ref.norm()).item() * 100
    print(f"  {name:<32} {fmt_err(err)}")

print(f"\n  Note: L2 drift is dominated by high-magnitude elements where bnb preserves")
print(f"  precision via absmax normalization. Update error (rsqrt path) weights all")
print(f"  elements equally in relative terms, exposing bnb's failure on low-magnitude")
print(f"  rows. The two metrics answer different questions.")

F2: Factored vs full-rank V (5000 steps, R=2048, C=2048, device=cuda)
  Row heterogeneity: 409 hot / 1639 cold, BR = 36.7 decades

  config                             V_drift%
  --------------------------------------------
  2D factored (log_adapt)             6.0797
  2D factored (bnb)                   5.0622
  1D full-rank (log_adapt)            3.2006
  1D full-rank (bnb)                  8.5148

  config                             upd_err%
  --------------------------------------------
  2D factored (log_adapt)              3.220
  2D factored (bnb)                    >100%
  1D full-rank (log_adapt)             1.605
  1D full-rank (bnb)                   3.803

  Note: L2 drift is dominated by high-magnitude elements where bnb preserves
  precision via absmax normalization. Update error (rsqrt path) weights all
  elements equally in relative terms, exposing bnb's failure on low-magnitude
  rows. The two metrics answer different questions.


In [30]:
# F3: Extreme-BR layer quantization (emb, lm_head)
# Distribution parameters from state trace (step 10000, Adafactor factored V):
#   emb row V:     log2 p5 = -95.1, p95 = -21.5, med = -28.5  (BR ~ 74 dec)
#   lm_head row V: log2 p5 = -50.9, p95 = -21.4, med = -29.2  (BR ~ 30 dec)
# Note: simulated BR (106/44 dec) slightly exceeds trace BR (82/35 dec)
# due to Gaussian tails; conclusion direction is unaffected.

n = 32000

distributions = {
    "emb-like (BR~74)":     lambda rng: torch.pow(2.0, torch.randn(n, generator=rng) * 12.0 - 28.0),
    "lm_head-like (BR~30)": lambda rng: torch.pow(2.0, torch.randn(n, generator=rng) * 5.0 - 29.0),
}

methods = [
    ("log_adapt bs=2048", lambda v: qd_log_adaptive(v, 2048)),
    ("log_adapt bs=512",  lambda v: qd_log_adaptive(v, 512)),
    ("log_adapt bs=256",  lambda v: qd_log_adaptive(v, 256)),
    ("log_adapt bs=128",  lambda v: qd_log_adaptive(v, 128)),
    ("bnb bs=256",        lambda v: qd_bnb(v, 256)),
]

print(f"F3: Extreme-BR layer quantization (n={n}, single step)")

for dname, dgen in distributions.items():
    rng = torch.Generator().manual_seed(42)
    v = dgen(rng)
    v_nz = v[v > 0]
    br = (torch.log2(v_nz.max()) - torch.log2(v_nz.min())).item()
    rng_g = torch.Generator().manual_seed(7)
    grad = torch.randn(n, generator=rng_g) * 0.01

    print(f"\n  {dname}, simulated BR = {br:.1f} decades")
    print(f"  {'method':<22} {'L1%':>8} {'upd_err%':>10}")
    print("  " + "-" * 42)
    for mname, mfn in methods:
        v_q = mfn(v)
        l1 = rel_l2(v, v_q) * 100
        ue = update_error_adafactor(grad, v, v_q) * 100
        print(f"  {mname:<22} {l1:>7.2f} {fmt_err(ue)}")

F3: Extreme-BR layer quantization (n=32000, single step)

  emb-like (BR~74), simulated BR = 105.7 decades
  method                      L1%   upd_err%
  ------------------------------------------
  log_adapt bs=2048         0.03     0.692
  log_adapt bs=512          0.01     0.135
  log_adapt bs=256          0.00     0.079
  log_adapt bs=128          0.00     0.018
  bnb bs=256                0.00     >100%

  lm_head-like (BR~30), simulated BR = 44.1 decades
  method                      L1%   upd_err%
  ------------------------------------------
  log_adapt bs=2048         0.48     1.003
  log_adapt bs=512          0.26     0.746
  log_adapt bs=256          0.16     0.628
  log_adapt bs=128          0.10     0.494
  bnb bs=256                0.05     >100%


In [4]:
# F4: Secondary factors
# F4a: RMS scaling feedback (does parameter drift affect RMS, which affects lr?)
# F4b: Dynamic beta (Adafactor step^-0.8) vs fixed beta
# F4c: Gradient noise level (batch size proxy)

n = 8192
eps1 = 1e-30
eps2 = 1e-3
lr = 1e-3
clip_threshold = 1.0
qfn_v = lambda v: qd_log_adaptive(v, 2048)

# --- F4a: RMS feedback (10000 steps) ---
steps_a = 10000
rng = torch.Generator().manual_seed(42)
grads_a = [torch.randn(n, generator=rng) * 0.01 for _ in range(steps_a)]
p_init = torch.randn(n, generator=torch.Generator().manual_seed(7)) * 0.02

def run_rms_test(freeze_rms):
    p_ref = p_init.clone(); v_ref = torch.zeros(n)
    p_q = p_init.clone();   v_q = torch.zeros(n)
    rms_traj = []
    for t, g in enumerate(grads_a):
        step = t + 1
        beta2t = 1.0 - step ** (-0.8)
        g_sq_eps = g.square() + eps1
        v_ref = beta2t * v_ref + (1 - beta2t) * g_sq_eps
        u_ref = g * torch.rsqrt(v_ref.clamp(min=MIN_VAL))
        rms_u = (u_ref.norm() / math.sqrt(n)).item()
        u_ref = u_ref / max(rms_u / clip_threshold, 1.0)
        rms_traj.append((p_ref.norm() / math.sqrt(n)).item())
        p_ref = p_ref - max(eps2, rms_traj[-1]) * lr * u_ref

        v_new = beta2t * v_q + (1 - beta2t) * g_sq_eps
        v_q = qfn_v(v_new)
        u_q = g * torch.rsqrt(v_q.clamp(min=MIN_VAL))
        rms_uq = (u_q.norm() / math.sqrt(n)).item()
        u_q = u_q / max(rms_uq / clip_threshold, 1.0)
        if freeze_rms:
            lr_eff = max(eps2, rms_traj[t]) * lr
        else:
            lr_eff = max(eps2, (p_q.norm() / math.sqrt(n)).item()) * lr
        p_q = p_q - lr_eff * u_q
    return ((p_q - p_ref).norm() / p_ref.norm()).item() * 100

drift_nat = run_rms_test(freeze_rms=False)
drift_frz = run_rms_test(freeze_rms=True)

print(f"F4a: RMS feedback ({steps_a} steps)")
print(f"  natural RMS:  {drift_nat:.4f}%")
print(f"  frozen RMS:   {drift_frz:.4f}%")
print(f"  feedback contribution: {drift_nat - drift_frz:+.4f}%")

# --- F4b: Dynamic beta vs fixed beta (10000 steps) ---
steps_b = 10000
print(f"\nF4b: Dynamic beta vs fixed beta ({steps_b} steps)")

for beta_mode, beta_fn in [("dynamic (step^-0.8)", lambda s: 1.0 - s ** (-0.8)),
                            ("fixed (beta2=0.999)", lambda s: 0.999)]:
    rng_b = torch.Generator().manual_seed(42)
    v = torch.zeros(n); v_ref = torch.zeros(n)
    report = []
    for t in range(steps_b):
        g = torch.randn(n, generator=rng_b) * 0.01
        step = t + 1
        beta2t = beta_fn(step)
        bvv = 1.0 - beta2t
        g_sq_eps = g.square() + eps1
        v_ref = beta2t * v_ref + bvv * g_sq_eps
        v = qfn_v(beta2t * v + bvv * g_sq_eps)
        if step in (2000, 4000, 6000, 8000, 10000):
            report.append((step, rel_l2(v_ref, v) * 100))
    print(f"\n  {beta_mode}")
    print(f"  {'step':<8} {'V_drift%':>10}")
    print("  " + "-" * 20)
    for s, e in report:
        print(f"  {s:<8} {e:>9.4f}")

# --- F4c: Gradient noise level (10000 steps) ---
steps_c = 10000
grad_stds = [0.005, 0.011, 0.022, 0.05]

print(f"\nF4c: Gradient noise level ({steps_c} steps)")
print(f"  {'std':<10} {'drift@5k%':>11} {'drift@10k%':>12}")
print("  " + "-" * 35)

for gstd in grad_stds:
    rng_c = torch.Generator().manual_seed(42)
    p_init_c = torch.randn(n, generator=torch.Generator().manual_seed(7)) * 0.02
    p_ref = p_init_c.clone(); v_ref = torch.zeros(n)
    p_q = p_init_c.clone();   v_q = torch.zeros(n)
    drifts = {}
    for t in range(steps_c):
        g = torch.randn(n, generator=rng_c) * gstd
        step = t + 1
        beta2t = 1.0 - step ** (-0.8)
        g_sq_eps = g.square() + eps1
        v_ref = beta2t * v_ref + (1 - beta2t) * g_sq_eps
        u_ref = g * torch.rsqrt(v_ref.clamp(min=MIN_VAL))
        rms_u = (u_ref.norm() / math.sqrt(n)).item()
        u_ref = u_ref / max(rms_u / clip_threshold, 1.0)
        p_ref = p_ref - max(eps2, (p_ref.norm() / math.sqrt(n)).item()) * lr * u_ref

        v_new = beta2t * v_q + (1 - beta2t) * g_sq_eps
        v_q = qfn_v(v_new)
        u_q = g * torch.rsqrt(v_q.clamp(min=MIN_VAL))
        rms_uq = (u_q.norm() / math.sqrt(n)).item()
        u_q = u_q / max(rms_uq / clip_threshold, 1.0)
        p_q = p_q - max(eps2, (p_q.norm() / math.sqrt(n)).item()) * lr * u_q

        if step in (5000, 10000):
            drifts[step] = ((p_q - p_ref).norm() / p_ref.norm()).item() * 100
    print(f"  {gstd:<10} {drifts[5000]:>10.4f} {drifts[10000]:>11.4f}")

F4a: RMS feedback (10000 steps)
  natural RMS:  0.0350%
  frozen RMS:   0.0349%
  feedback contribution: +0.0000%

F4b: Dynamic beta vs fixed beta (10000 steps)

  dynamic (step^-0.8)
  step       V_drift%
  --------------------
  2000        0.6242
  4000        0.6868
  6000        0.8101
  8000        0.9542
  10000       1.0851

  fixed (beta2=0.999)
  step       V_drift%
  --------------------
  2000        0.7554
  4000        0.8198
  6000        0.8096
  8000        0.7939
  10000       0.7499

F4c: Gradient noise level (10000 steps)
  std          drift@5k%   drift@10k%
  -----------------------------------
  0.005          0.0216      0.0357
  0.011          0.0215      0.0356
  0.022          0.0215      0.0356
  0.05           0.0213      0.0348


In [32]:
# F5: CAME residual under extreme BR — memory optimized
# Runs fp32 reference and all quantized configs in a single pass
# to avoid storing 10000 steps of 2048x2048 tensors (which caused OOM).

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
R, C = 2048, 2048
steps = 10000
beta1, beta2, beta3 = 0.9, 0.999, 0.9999
eps1, eps_came = 1e-30, 1e-16
clip_threshold = 1.0
m_bs = 256

# Extreme BR distribution (matching emb/lm_head trace)
rng_init = torch.Generator(device=device).manual_seed(42)
row_log_scale = torch.zeros(R, device=device)
n_hot = int(R * 0.2)
row_log_scale[:n_hot] = torch.randn(n_hot, generator=rng_init, device=device) * 1.0 - 10.0
row_log_scale[n_hot:] = torch.randn(R - n_hot, generator=rng_init, device=device) * 1.0 - 40.0
row_scale = torch.pow(2.0, row_log_scale)
br = (torch.log2(row_scale.max()) - torch.log2(row_scale.min())).item()

def asg(row, col):
    return (row / row.mean().clamp(min=eps1)).rsqrt().unsqueeze(-1) * col.unsqueeze(-2).rsqrt()

# Configurations: (name, v_qfn, m_qfn, use_bc)
configs = {
    "la+fp32M":       (lambda v: qd_log_adaptive(v, 2048), None,                                  False),
    "la+UF8M":        (lambda v: qd_log_adaptive(v, 2048), lambda m: qd_uf(m, m_bs, 8),           False),
    "la+D8M":         (lambda v: qd_log_adaptive(v, 2048), lambda m: qd_dynamic_signed(m, m_bs, 8), False),
    "la+UF8M+BC":     (lambda v: qd_log_adaptive(v, 2048), lambda m: qd_uf(m, m_bs, 8),           True),
    "bnb+fp32M":      (lambda v: qd_bnb(v, 256),           None,                                  False),
}

# Adaptive windows
n_windows = 5
ws = steps // n_windows
windows = [(i * ws, (i + 1) * ws) for i in range(n_windows)]

# Initialize states
rng_loop = torch.Generator(device=device).manual_seed(42)

# fp32 states
row_ref = torch.zeros(R, device=device); col_ref = torch.zeros(C, device=device)
m_ref = torch.zeros(R, C, device=device)
rr_ref = torch.zeros(R, device=device); rc_ref = torch.zeros(C, device=device)

# quantized states
q_states = {}
for cname in configs:
    q_states[cname] = {
        "row": torch.zeros(R, device=device), "col": torch.zeros(C, device=device),
        "m": torch.zeros(R, C, device=device),
        "rr": torch.zeros(R, device=device), "rc": torch.zeros(C, device=device),
        "errors": [], "res_biases": [],
        "nan_consecutive": 0, "stopped": False, "nan_window": -1
    }

print(f"F5: CAME residual under extreme BR ({steps} steps, R={R}, C={C}, device={device})")
print(f"  Row heterogeneity: BR = {br:.1f} decades")

for t in range(steps):
    step = t + 1
    g = torch.randn(R, C, generator=rng_loop, device=device) * 0.01 * row_scale.unsqueeze(1)
    g_sq = g.square() + eps1
    
    # --- fp32 reference ---
    row_ref = beta2 * row_ref + (1 - beta2) * g_sq.mean(dim=-1)
    col_ref = beta2 * col_ref + (1 - beta2) * g_sq.mean(dim=0)
    u_t_ref = asg(row_ref, col_ref) * g
    rms_u_ref = (u_t_ref.norm() / math.sqrt(R * C)).item()
    u_t_ref = u_t_ref / max(rms_u_ref / clip_threshold, 1.0)
    m_ref = beta1 * m_ref + (1 - beta1) * u_t_ref
    res_ref = (u_t_ref - m_ref).square() + eps_came
    rr_ref = beta3 * rr_ref + (1 - beta3) * res_ref.mean(dim=-1)
    rc_ref = beta3 * rc_ref + (1 - beta3) * res_ref.mean(dim=0)
    u_final_ref = asg(rr_ref, rc_ref) * m_ref
    res_ref_mean = res_ref.mean().item()
    
    # --- quantized configs ---
    for cname, (v_qfn, m_qfn, use_bc) in configs.items():
        st = q_states[cname]
        if st["stopped"]:
            st["errors"].append(float('nan'))
            continue
            
        row_q = v_qfn(beta2 * st["row"] + (1 - beta2) * g_sq.mean(dim=-1))
        col_q = v_qfn(beta2 * st["col"] + (1 - beta2) * g_sq.mean(dim=0))
        u_t_q = asg(row_q, col_q) * g
        rms_u_q = (u_t_q.norm() / math.sqrt(R * C)).item()
        u_t_q = u_t_q / max(rms_u_q / clip_threshold, 1.0)
        
        m_new = beta1 * st["m"] + (1 - beta1) * u_t_q
        m_q = m_qfn(m_new) if m_qfn else m_new
        st["m"] = m_q
        
        res_q = (u_t_q - m_q).square() + eps_came
        if res_ref_mean > 1e-30:
            st["res_biases"].append((res_q.mean().item() / res_ref_mean) - 1.0)
            
        st["rr"] = v_qfn(beta3 * st["rr"] + (1 - beta3) * res_q.mean(dim=-1))
        st["rc"] = v_qfn(beta3 * st["rc"] + (1 - beta3) * res_q.mean(dim=0))
        u_final_q = asg(st["rr"], st["rc"]) * m_q
        
        if use_bc:
            bc1 = 1.0 - beta1 ** step
            bc2 = 1.0 - beta2 ** step
            u_final_q = u_final_q * (math.sqrt(bc2) / bc1)
            
        # NaN detection
        if torch.isnan(u_final_q).any() or torch.isinf(u_final_q).any():
            st["nan_consecutive"] += 1
            st["errors"].append(float('nan'))
            if st["nan_consecutive"] >= 3:
                st["stopped"] = True
                st["nan_window"] = t // ws
        else:
            st["nan_consecutive"] = 0
            ref_n = u_final_ref.norm()
            err = ((u_final_q - u_final_ref).norm() / ref_n).item() if ref_n > 1e-30 else 0.0
            st["errors"].append(err)
            
        # Update states for next step
        st["row"] = row_q
        st["col"] = col_q

# --- Report ---
print(f"  Residual EMA convergence at step {steps}: {(1 - beta3**steps)*100:.1f}%")
print(f"  M quantization: UF8 bs={m_bs} (where applicable)")

print(f"\n  {'window':<14}", end="")
for cname in configs:
    print(f" {cname:>16}", end="")
print()
print("  " + "-" * (14 + 17 * len(configs)))

for wi, (s, e) in enumerate(windows):
    print(f"  {s:>5}-{e:<5}  ", end="")
    for cname in configs:
        st = q_states[cname]
        if st["stopped"] and wi > st["nan_window"]:
            print(f" {'--':>16}", end="")
        else:
            w = st["errors"][s:e]
            if any(math.isnan(x) for x in w):
                print(f" {'nan':>16}", end="")
            else:
                mean_e = sum(w) / len(w) * 100
                print(f" {mean_e:>15.4f}%", end="")
    print()

print(f"\n  {'config':<18} {'res_bias%':>12}")
print("  " + "-" * 32)
for cname in configs:
    st = q_states[cname]
    rb = (sum(st["res_biases"]) / len(st["res_biases"]) * 100) if st["res_biases"] else float('nan')
    rb_s = f"{rb:>+11.4f}" if not math.isnan(rb) else f"{'nan':>11}"
    print(f"  {cname:<18} {rb_s}")

print(f"\n  Note: res_bias = mean(res_quant / res_fp32 - 1).")
print(f"  res_bias is similar across fp32-M and quantized-M configs,")
print(f"  indicating the positive bias originates from V quantization")
print(f"  (delta_V^2 term), not from M quantization.")

F5: CAME residual under extreme BR (10000 steps, R=2048, C=2048, device=cuda)
  Row heterogeneity: BR = 36.7 decades
  Residual EMA convergence at step 10000: 63.2%
  M quantization: UF8 bs=256 (where applicable)

  window                 la+fp32M          la+UF8M           la+D8M       la+UF8M+BC        bnb+fp32M
  ---------------------------------------------------------------------------------------------------
      0-2000             0.4200%          1.6812%          2.5235%         27.2597%              nan
   2000-4000             0.4138%          1.6804%          2.5261%          3.5404%               --
   4000-6000             0.4368%          1.6867%          2.5312%          1.7042%               --
   6000-8000             0.4199%          1.6801%          2.5275%          1.6775%               --
   8000-10000            0.3986%          1.6758%          2.5241%          1.6756%               --

  config                res_bias%
  --------------------------------
  la+fp

In [3]:
# F6: Quick Test 8-bit vs 16-bit adaptive log V
# Exploring the theoretical upper bound of log-space V quantization.

n = 32000
bs = 2048

distributions = {
    "medium_10dec":         lambda rng: torch.pow(2.0, torch.randn(n, generator=rng) * 3.0 - 15.0),
    "lm_head-like (BR~30)": lambda rng: torch.pow(2.0, torch.randn(n, generator=rng) * 5.0 - 29.0),
    "emb-like (BR~74)":     lambda rng: torch.pow(2.0, torch.randn(n, generator=rng) * 12.0 - 28.0),
}

methods = {
    "log_adapt 8-bit (255+z)":    lambda v: qd_log_adaptive_generic(v, bs, 256, zero_reserved=True),
    "log_adapt 16-bit (65535+z)": lambda v: qd_log_adaptive_generic(v, bs, 65536, zero_reserved=True),
}

print("F6: 8-bit vs 16-bit adaptive log V (Upper Bound Exploration)")
print(f"  n={n}, bs={bs}\n")

for dname, dgen in distributions.items():
    rng = torch.Generator().manual_seed(42)
    v = dgen(rng)
    v_nz = v[v > 0]
    br = (torch.log2(v_nz.max()) - torch.log2(v_nz.min())).item() if v_nz.numel() > 0 else 0.0
    
    rng_g = torch.Generator().manual_seed(7)
    grad = torch.randn(n, generator=rng_g) * 0.01
    
    print(f"  {dname} (simulated BR = {br:.1f} decades)")
    print(f"  {'method':<30} {'L1%':>8} {'upd_err%':>10}")
    print("  " + "-" * 50)
    
    for mname, mfn in methods.items():
        v_q = mfn(v)
        l1 = rel_l2(v, v_q) * 100
        ue = update_error_adafactor(grad, v, v_q) * 100
        print(f"  {mname:<30} {l1:>7.4f} {fmt_err(ue)}")
    print()

# Memory comparison
print("  Memory overhead for 1.1B params (V only):")
print(f"    8-bit:  {1.1e9 * (8 + 64/bs) / 8 / 1024 / 1024:.1f} MB")
print(f"    16-bit: {1.1e9 * (16 + 64/bs) / 8 / 1024 / 1024:.1f} MB  (+1100 MB)")
print(f"    FP16:   {1.1e9 * 16 / 8 / 1024 / 1024:.1f} MB  (Native, no metadata)")

F6: 8-bit vs 16-bit adaptive log V (Upper Bound Exploration)
  n=32000, bs=2048

  medium_10dec (simulated BR = 26.4 decades)
  method                              L1%   upd_err%
  --------------------------------------------------
  log_adapt 8-bit (255+z)         0.9706     0.740
  log_adapt 16-bit (65535+z)      0.0040     0.003

  lm_head-like (BR~30) (simulated BR = 44.1 decades)
  method                              L1%   upd_err%
  --------------------------------------------------
  log_adapt 8-bit (255+z)         0.4797     1.003
  log_adapt 16-bit (65535+z)      0.0020     0.004

  emb-like (BR~74) (simulated BR = 105.7 decades)
  method                              L1%   upd_err%
  --------------------------------------------------
  log_adapt 8-bit (255+z)         0.0341     0.692
  log_adapt 16-bit (65535+z)      0.0003     0.002

  Memory overhead for 1.1B params (V only):
    8-bit:  1053.1 MB
    16-bit: 2102.2 MB  (+1100 MB)
    FP16:   2098.1 MB  (Native, no metadata)

In [9]:
# F7: Trace-calibrated median-range sensitivity
#
# Controlled synthetic blocks calibrated to the official CAME trace at step 10K.
# Targets: tensor-wise median log2 value + median 2048-element block log2 range.
# Attention/MLP targets are medians across parameter-state tensors.
# Embedding/LM-head targets are singleton tensor observations.
#
# These are controlled synthetic blocks, not empirical trace samples.
# EMA accumulation effects are evaluated separately in F5.

BLOCK_SIZE = 2048
N_BLOCKS = 16
n = BLOCK_SIZE * N_BLOCKS  # 32768; no partial block


def make_range_matched_blocks(
    log2_median,
    target_block_range,
    seed=42,
):
    """Create blocks with exact median and log2 min-to-max range."""
    rng = torch.Generator().manual_seed(seed)
    z = torch.randn(N_BLOCKS, BLOCK_SIZE, generator=rng)

    # Center every block on its median.
    z = z - z.median(dim=1, keepdim=True).values

    # Rescale every block to the requested log2 range.
    span = (
        z.max(dim=1, keepdim=True).values
        - z.min(dim=1, keepdim=True).values
    )
    log2_x = log2_median + z * (target_block_range / span)

    return torch.pow(2.0, log2_x).flatten()


def median_block_log2_range(v, block_size=BLOCK_SIZE):
    blocks, _ = _pad_blocks(v.flatten(), block_size)
    log2_v = torch.log2(blocks.clamp(min=MIN_VAL))
    block_ranges = (
        log2_v.max(dim=1).values
        - log2_v.min(dim=1).values
    )
    return block_ranges.median().item()


def format_metric(value):
    if not math.isfinite(value):
        return "inf"
    if value > 100:
        return ">100"
    return f"{value:.3f}"


# Source: official CAME trace, step 10K.
# Tuple: label, tensor-wise median log2 value, median block log2 range.
trace_cases = [
    ("V_r attention", -29.414945,  3.948610),
    ("V_r MLP",       -35.247841, 17.454587),
    ("V_r embedding", -28.959490, 84.255028),
    ("C_r attention",  -1.471986,  0.877168),
    ("C_r MLP",        -2.256184,  8.241929),
    ("C_r LM head",    -2.700284, 12.249847),
    ("C_r embedding",  -2.938479, 53.298794),
]

methods = [
    (
        "AL8, B=2048",
        lambda v: qd_log_adaptive(v, 2048),
    ),
    (
        "AL16, B=2048",
        lambda v: qd_log_adaptive_generic(
            v,
            2048,
            65536,
            zero_reserved=True,
        ),
    ),
    (
        "Dyn8, B=256",
        lambda v: qd_bnb(v, 256),
    ),
]

rng_g = torch.Generator().manual_seed(7)
grad = torch.randn(n, generator=rng_g) * 0.01
grad64 = grad.double()

print(
    f"F7: trace-calibrated median-range sensitivity "
    f"(n={n}, single step)"
)
print("  Targets derived from the official CAME trace at step 10K")
print(
    "  Synthetic blocks match the traced median value and "
    "median 2048-element block log2 range"
)
print(
    "  Attention/MLP use across-tensor medians; "
    "Embedding/LM head are singleton observations"
)
print("  These are controlled synthetic blocks, not empirical trace samples")
print("  Note: statewise probe; EMA accumulation is evaluated in F5")

for name, target_median, target_range in trace_cases:
    dist = make_range_matched_blocks(
        target_median,
        target_range,
    )

    realized_median = torch.log2(dist).median().item()
    realized_range = median_block_log2_range(dist)

    # Guard against accidentally reverting to an unconstrained distribution.
    assert math.isclose(
        realized_median,
        target_median,
        abs_tol=1e-3,
    )
    assert math.isclose(
        realized_range,
        target_range,
        abs_tol=1e-3,
    )

    print(
        f"\n  {name}: "
        f"target median={target_median:.3f}, "
        f"realized median={realized_median:.3f}; "
        f"target BR={target_range:.3f}, "
        f"realized BR={realized_range:.3f} bits"
    )
    print(
        f"  {'method':<18} "
        f"{'rel_L2%':>9} "
        f"{'inv_std%':>10} "
        f"{'upd_err%':>10} "
        f"{'false0':>8}"
    )
    print("  " + "-" * 62)

    # Compute diagnostics in FP64 to prevent norm overflow.
    dist64 = dist.double()
    inv_ref = torch.rsqrt(dist64.clamp(min=MIN_VAL))
    update_ref = grad64 * inv_ref

    for method_name, quantize in methods:
        dist_q = quantize(dist)
        dist_q64 = dist_q.double()

        rel_l2_error = (
            (dist_q64 - dist64).norm()
            / dist64.norm()
        ).item() * 100

        inv_approx = torch.rsqrt(
            dist_q64.clamp(min=MIN_VAL)
        )
        inv_error = (
            (inv_approx - inv_ref).norm()
            / inv_ref.norm()
        ).item() * 100

        update_approx = grad64 * inv_approx
        update_error = (
            (update_approx - update_ref).norm()
            / update_ref.norm()
        ).item() * 100

        false_zeros = int(
            ((dist > 0) & (dist_q == 0)).sum().item()
        )

        print(
            f"  {method_name:<18} "
            f"{rel_l2_error:>9.4f} "
            f"{format_metric(inv_error):>10} "
            f"{format_metric(update_error):>10} "
            f"{false_zeros:>8d}"
        )

F7: trace-calibrated median-range sensitivity (n=32768, single step)
  Targets derived from the official CAME trace at step 10K
  Synthetic blocks match the traced median value and median 2048-element block log2 range
  Attention/MLP use across-tensor medians; Embedding/LM head are singleton observations
  These are controlled synthetic blocks, not empirical trace samples
  Note: statewise probe; EMA accumulation is evaluated in F5

  V_r attention: target median=-29.415, realized median=-29.415; target BR=3.949, realized BR=3.949 bits
  method               rel_L2%   inv_std%   upd_err%   false0
  --------------------------------------------------------------
  AL8, B=2048           0.3113      0.156      0.154        0
  AL16, B=2048          0.0012      0.001      0.001        0
  Dyn8, B=256           0.5425      0.434      0.434        0

  V_r MLP: target median=-35.248, realized median=-35.248; target BR=17.455, realized BR=17.455 bits
  method               rel_L2%   inv_std%  

## §G Appendix

G1: Cold-word first-hit (Adafactor algorithm property).
G2: Sign correction at -0.0 boundary (bnb vs ours).
G3: MIN_VAL clamp redundancy under realistic training.

In [34]:
# G1: Cold-word first-hit
# When V=0 and the first gradient arrives, update = sign(g) / sqrt(beta_val).
# This is an Adafactor algorithm property, independent of quantization.

print("G1: Cold-word first-hit magnitude (Adafactor, beta2_decay=-0.8)")
print(f"\n  {'step':<8} {'beta_val':<12} {'|update|':<12} {'rms (N=1M)':<14} {'clipped?':<10}")
print("  " + "-" * 58)
for step in [1, 10, 100, 1000, 10000]:
    bv = step ** (-0.8)
    upd = 1.0 / math.sqrt(bv)
    rms = upd / math.sqrt(1_000_000)
    clipped = "yes" if rms > 1.0 else "no"
    print(f"  {step:<8} {bv:<12.6f} {upd:<12.2f} {rms:<14.6f} {clipped:<10}")

G1: Cold-word first-hit magnitude (Adafactor, beta2_decay=-0.8)

  step     beta_val     |update|     rms (N=1M)     clipped?  
  ----------------------------------------------------------
  1        1.000000     1.00         0.001000       no        
  10       0.158489     2.51         0.002512       no        
  100      0.025119     6.31         0.006310       no        
  1000     0.003981     15.85        0.015849       no        
  10000    0.000631     39.81        0.039811       no        


In [35]:
# G2: Sign correction at -0.0 boundary
# bnb: always corrects sign mismatch. For x=-0.0, pushes index to negative side.
# Ours: skips correction when x == 0.0f. No practical impact.

qmap = BNB_S8

def quant_sign_mode(v_flat, absmax, mode):
    x_norm = v_flat / absmax
    idx = torch.searchsorted(qmap, x_norm.contiguous()).clamp(0, 255)
    prev = (idx - 1).clamp(0)
    idx = torch.where((x_norm - qmap[prev]).abs() < (x_norm - qmap[idx]).abs(), prev, idx)
    if mode == "none":
        return idx
    map_neg = torch.signbit(qmap[idx])
    val_neg = torch.signbit(v_flat)
    mismatch = map_neg != val_neg
    if mode == "ours":
        mismatch = mismatch & (v_flat != 0.0)
    pos_fix = mismatch & (v_flat > 0)
    neg_fix = mismatch & ~(v_flat > 0) if mode == "bnb" else mismatch & (v_flat < 0)
    idx = torch.where(pos_fix, (idx + 1).clamp(max=255), idx)
    idx = torch.where(neg_fix, (idx - 1).clamp(min=0), idx)
    return idx

m_test = torch.zeros(256)
m_test[0:5] = 0.0
m_test[5:10] = torch.tensor(-0.0)
m_test[10:15] = 1e-10
m_test[15:20] = -1e-10
m_test[20:] = torch.randn(236) * 0.001
absmax = m_test.abs().max().clamp(min=1e-12)

idx_ours = quant_sign_mode(m_test, absmax, "ours")
idx_bnb = quant_sign_mode(m_test, absmax, "bnb")
diff = (idx_ours != idx_bnb).sum().item()

print("G2: Sign correction at -0.0 boundary")
print(f"  Elements where ours != bnb: {diff}/256")
if diff > 0:
    disagree = (idx_ours != idx_bnb).nonzero(as_tuple=True)[0]
    for i in disagree[:5]:
        print(f"    [{i.item()}] val={m_test[i].item():+.1e}  "
              f"ours_q={idx_ours[i].item()} (deq={qmap[idx_ours[i]].item():+.4e})  "
              f"bnb_q={idx_bnb[i].item()} (deq={qmap[idx_bnb[i]].item():+.4e})")
print(f"\n  Affected values: only -0.0 (IEEE negative zero).")
print(f"  Error magnitude: {abs(qmap[126].item()):.2e} * absmax (negligible).")

G2: Sign correction at -0.0 boundary
  Elements where ours != bnb: 5/256
    [5] val=-0.0e+00  ours_q=127 (deq=+0.0000e+00)  bnb_q=126 (deq=-5.5000e-07)
    [6] val=-0.0e+00  ours_q=127 (deq=+0.0000e+00)  bnb_q=126 (deq=-5.5000e-07)
    [7] val=-0.0e+00  ours_q=127 (deq=+0.0000e+00)  bnb_q=126 (deq=-5.5000e-07)
    [8] val=-0.0e+00  ours_q=127 (deq=+0.0000e+00)  bnb_q=126 (deq=-5.5000e-07)
    [9] val=-0.0e+00  ours_q=127 (deq=+0.0000e+00)  bnb_q=126 (deq=-5.5000e-07)

  Affected values: only -0.0 (IEEE negative zero).
  Error magnitude: 5.50e-07 * absmax (negligible).


In [36]:
# G3: MIN_VAL clamp redundancy
# Simulates EMA with adaptive log quantization, checks if v_new ever falls below MIN_VAL.
# Synthetic EMA configurations; does not claim to represent specific training runs.

print("G3: MIN_VAL clamp redundancy (5000-step EMA)")

scenarios = [
    # (name, grad_std, sparsity, eps_added_to_gsq)
    ("standard",        0.01,   0.9,  1e-30),
    ("large eps",       0.01,   0.9,  1e-8),
    ("small grad",      0.001,  0.9,  1e-30),
    ("dense",           0.01,   0.0,  1e-30),
    ("high beta2",      0.01,   0.9,  1e-30),
    ("very small grad", 0.0001, 0.9,  1e-30),
]

for sname, grad_std, sparsity, eps_sq in scenarios:
    n = 8192
    beta2_val = 0.9995 if sname == "high beta2" else 0.999
    beta_val = 1.0 - beta2_val
    rng = torch.Generator().manual_seed(42)
    cold_mask = torch.rand(n, generator=rng) < sparsity
    v = torch.zeros(n)
    min_nz = float('inf')
    below_count = 0
    for _ in range(5000):
        g = torch.randn(n, generator=rng) * grad_std
        g[cold_mask] = 0.0
        v = beta2_val * v + beta_val * (g.square() + eps_sq)
        v_nz = v[v > 0]
        if v_nz.numel() > 0:
            cur_min = v_nz.min().item()
            min_nz = min(min_nz, cur_min)
            if cur_min < MIN_VAL:
                below_count += 1
    print(f"  {sname:<18} min(v>0)={min_nz:.2e}  MIN_VAL={MIN_VAL:.2e}  "
          f"below={below_count}/5000  {'redundant' if below_count == 0 else 'NEEDED'}")

G3: MIN_VAL clamp redundancy (5000-step EMA)
  standard           min(v>0)=1.00e-33  MIN_VAL=1.18e-38  below=0/5000  redundant
  large eps          min(v>0)=1.00e-11  MIN_VAL=1.18e-38  below=0/5000  redundant
  small grad         min(v>0)=1.00e-33  MIN_VAL=1.18e-38  below=0/5000  redundant
  dense              min(v>0)=4.77e-16  MIN_VAL=1.18e-38  below=0/5000  redundant
  high beta2         min(v>0)=5.00e-34  MIN_VAL=1.18e-38  below=0/5000  redundant
  very small grad    min(v>0)=1.00e-33  MIN_VAL=1.18e-38  below=0/5000  redundant
